<a href="https://colab.research.google.com/github/bogdanparvu18/msc-graduate-project/blob/main/notebooks/phase2/phase2_05_finding_segments_and_temporal_supervision.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 2 — Sector 5: Finding Segments and Temporal Supervision

This notebook groups temporally adjacent verified frames of the same finding class within each video into finding segments. It loads validated artifacts from Sectors 2-4, preserves the physical-frame train, validation, and test assignments, and creates reproducible segment identifiers. Each segment is summarized in a manifest containing its observed frame range, verified frame count, and representative frame with associated image and bounding-box metadata. The sector also reports class support across splits and characterizes temporal supervision, without treating observed segment limits as exact clinical onset or offset boundaries.
This notebook reads published artifacts from Sectors 2, 3 and 4 at one pinned GitHub main commit. It does not execute upstream notebooks or load raw media.


Sections 6–10 define the pure transformations. Section 11 computes their results once and binds the tables to the processing signature before validation. Section 12 rejects stale or modified results before saving.


### 1. Imports and dependencies

 Pandas, NumPy and PyArrow are the only runtime dependencies; the supplied Sector 5 transformations run on CPU.

In [1]:
%pip install -q pandas numpy pyarrow


In [2]:
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
import copy
import hashlib
import json
import re
from types import CodeType, FunctionType
import math
import importlib.metadata
import numpy as np
import pandas as pd
from IPython.display import display
import shutil
import tempfile
import os
import subprocess


### 2. Published GitHub source and Sector 5 configuration

`repo_url` and `branch="main"` select the published upstream metadata. Each load fixes one
commit for the complete S2 → S3 → S4 chain. Local upstream-directory overrides are rejected.
`storage_root` controls S5 outputs/cache and live media paths, not upstream metadata authority.
`None` algorithm settings inherit S4 values, with the declared S5 defaults used when absent.


In [3]:
BOOTSTRAP = {
    "repo_url": "https://github.com/bogdanparvu18/msc-graduate-project.git",
    "branch": "main",
    "storage_root": "/content/drive/MyDrive/MMVQA_Clinical",
    "mount_google_drive": True,
}

SECTOR5_SETTINGS = {
    "segment_max_gap_frames": None,
    "minimum_verified_segment_frames": None,
    "local_output_dir": "outputs/phase2/sector5",
    "export_dir": "outputs/phase2/sector5",
    "push": True,
    "dry_run": False,
    "github_overrides": {
        # "token_secret_name": "GITHUB_TOKEN",  # Otherwise inherited from Sector 4.
    },
}

SECTOR5_DEFAULTS = {
    "segment_max_gap_frames": 1,
    "minimum_verified_segment_frames": 2,
}

### 3. Read and validate the published upstream chain

Catalogs, configurations, schemas and tables come only from Git objects at one freshly fetched
`main` commit. Validate the S2 export hashes (including compressed transport), S3/S4 catalogs,
producer links and raw reference identities before using any annotations. A private temporary
metadata workspace stays alive through saving so the pre-save checksum gate can recheck it.
No media files are loaded, and source errors never fall back to Drive.


In [4]:
# Read-only catalog helpers copied from the current Sector 4 reader.
# Their canonical hashes and dtype restoration match the existing Sector 3/4 files.
P3_NULL = "\\N"

S3_EXECUTION_METADATA_KEYS = frozenset({
    "run_id", "run_context", "run_datetime", "run_timestamp", "run_date", "run_time",
    "timestamp", "date", "time", "execution_date", "execution_time",
    "created_at", "created_at_utc", "created_utc", "saved_at", "saved_at_utc", "saved_utc",
    "started_at", "started_at_utc", "started_utc", "finished_at", "finished_at_utc", "finished_utc",
    "generated_at", "generated_at_utc", "generated_utc", "updated_at", "updated_at_utc", "updated_utc",
    "modified_at", "modified_at_utc", "modified_utc",
    "source_commit", "source_commit_sha", "source_commit_hash", "source_revision", "config_source_commit",
    "commit", "commit_sha", "head_commit", "current_commit", "repository_commit",
    "publication_receipt", "receipt", "last_publication",
    "source_configs", "config_provenance", "config_conflict_report", "lineage",
    "source_notebooks", "source_config_provenance", "config_sources", "literal_notebooks", "literal_configs",
    "input_sha256", "state_sha256", "upstream_states", "upstream_alignment_run_id",
    "notebook_path", "notebook_sha256", "source_notebook_path", "source_notebook_sha256",
    "source_config_path", "source_config_sha256", "source_snapshot_path", "source_snapshot_sha256",
    "config_snapshot_path", "config_snapshot_sha256", "snapshot_path", "snapshot_sha256",
    "source_config_file", "source_config_hash", "source_snapshot_file", "source_snapshot_hash",
    "config_snapshot_file", "config_snapshot_hash", "snapshot_file", "snapshot_hash",
    "config_inheritance_policy", "taxonomy_validation_policy",
})

def _s3_semantic_json_value(value):
    """Pure conversion to canonical JSON-compatible values; no I/O or clock access."""
    if value is pd.NA or value is pd.NaT:
        return None
    if isinstance(value, dict):
        return {str(key): _s3_semantic_json_value(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [_s3_semantic_json_value(item) for item in value]
    if isinstance(value, (set, frozenset)):
        values = [_s3_semantic_json_value(item) for item in value]
        return sorted(values, key=lambda item: json.dumps(item, sort_keys=True))
    if isinstance(value, (Path, PurePosixPath)):
        return str(value)
    if isinstance(value, np.generic):
        return _s3_semantic_json_value(value.item())
    if isinstance(value, float) and not np.isfinite(value):
        raise ValueError("Semantic configuration must not contain NaN or infinite values.")
    if value is None or isinstance(value, (str, bool, int, float)):
        return value
    raise TypeError(f"Unsupported semantic configuration value: {type(value).__name__}")

def _s3_semantic_json(value):
    return json.dumps(_s3_semantic_json_value(value), ensure_ascii=False, sort_keys=True,
                      separators=(",", ":"), allow_nan=False)

def _s3_semantic_digest(value):
    return hashlib.sha256(_s3_semantic_json(value).encode("utf-8")).hexdigest()

def sector3_semantic_config(config):
    """Project authored effective values, excluding execution/source bookkeeping only.

    Full source snapshots and current execution metadata remain available in the
    saved envelope. Authored paths, thresholds, publication settings (including
    push/dry_run), and unknown substantive config keys retain their meaning.
    """
    if not isinstance(config, dict):
        raise TypeError("Sector 3 semantic configuration requires a dictionary.")
    def project(value):
        if isinstance(value, dict):
            return {str(key): project(item) for key, item in value.items()
                    if str(key).lower() not in S3_EXECUTION_METADATA_KEYS}
        if isinstance(value, (list, tuple)):
            return [project(item) for item in value]
        return _s3_semantic_json_value(value)
    return project(config)

def sector3_semantic_config_fingerprint(config):
    """Stable identity of authored configuration, independent of execution time."""
    return _s3_semantic_digest(sector3_semantic_config(config))

def _p3_json_default(value):
    if value is pd.NA or value is pd.NaT:
        return None
    if isinstance(value, pd.DataFrame):
        return {"type": "dataframe", "columns": list(value.columns),
                "dtypes": {str(column): str(dtype) for column, dtype in value.dtypes.items()},
                "records": value.astype(object).where(value.notna(), None).to_dict("records")}
    if isinstance(value, (Path, PurePosixPath, datetime, pd.Timestamp)):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, (set, tuple)):
        return sorted(value) if isinstance(value, set) else list(value)
    raise TypeError(f"Unsupported Sector 3 JSON value: {type(value).__name__}")

def _p3_json_text(value):
    return json.dumps(value, sort_keys=True, indent=2, ensure_ascii=False,
                      allow_nan=False, default=_p3_json_default) + "\n"

def _p3_json_hash(value):
    return hashlib.sha256(_p3_json_text(value).encode("utf-8")).hexdigest()

def _p3_file_hash(path):
    with Path(path).open("rb") as source:
        return hashlib.file_digest(source, "sha256").hexdigest()

def _p3_relative(value):
    path = PurePosixPath(str(value))
    if path.is_absolute() or ".." in path.parts or len(path.parts) != 2:
        raise ValueError(f"Sector 3 artifacts must be flat relative paths: {value!r}")
    if path.parts[0] not in {"configs", "results", "reports"}:
        raise ValueError(f"Unexpected Sector 3 artifact group: {value!r}")
    return path.as_posix()

def _p3_safe_path(root, relative):
    path = root / _p3_relative(relative)
    if path.is_symlink() or root not in path.resolve().parents or path.parent.is_symlink():
        raise RuntimeError(f"Unsafe Sector 3 artifact path: {relative}")
    return path

def _p3_schema(frame):
    columns = []
    for column, dtype in frame.dtypes.items():
        descriptor = {"name": column, "dtype": str(dtype)}
        if isinstance(dtype, pd.StringDtype):
            descriptor["string_storage"] = dtype.storage
        if str(dtype) == "object":
            descriptor["object_kind"] = pd.api.types.infer_dtype(frame[column], skipna=True)
        if isinstance(dtype, pd.CategoricalDtype):
            descriptor.update({"categories": list(dtype.categories), "ordered": dtype.ordered})
        columns.append(descriptor)
    return {"rows": len(frame), "columns": columns, "column_axis_name": frame.columns.name,
            "csv_null_marker": P3_NULL, "schema_version": 1}

def sector3_table_fingerprint(frame):
    """Hash every value and its dtype schema; no clock or execution metadata is added."""
    if frame.columns.duplicated().any() or not all(isinstance(column, str) for column in frame.columns):
        raise ValueError("Sector 3 tables require unique string column names.")
    normalized = frame.reset_index(drop=True)
    digest = hashlib.sha256(_p3_json_text(_p3_schema(normalized)).encode("utf-8"))
    try:
        # Missing values have one semantic identity, including None/NaN/pd.NA in object columns.
        hash_values = normalized.astype(object).where(normalized.notna(), None)
        values = pd.util.hash_pandas_object(hash_values, index=False, categorize=False)
    except TypeError as error:
        raise TypeError("Sector 3 tables must contain scalar values; canonicalize nested conflict values first.") from error
    digest.update(values.to_numpy(dtype="uint64").astype("<u8", copy=False).tobytes())
    return digest.hexdigest()

def _p3_restore_csv(path, schema):
    frame = pd.read_csv(path, dtype="string", keep_default_na=False)
    expected = [descriptor["name"] for descriptor in schema["columns"]]
    if list(frame.columns) != expected or len(frame) != schema["rows"]:
        raise RuntimeError(f"CSV dimensions differ from saved schema: {path.name}")
    for descriptor in schema["columns"]:
        name, dtype = descriptor["name"], descriptor["dtype"]
        values = frame[name]
        nulls = values.eq(P3_NULL)
        values = values.mask(nulls, pd.NA)
        if dtype == "object":
            kind = descriptor.get("object_kind", "string")
            if kind in {"integer", "floating", "mixed-integer-float", "decimal"}:
                values = pd.to_numeric(values, errors="raise").astype(object)
            elif kind == "boolean":
                values = values.map({"True": True, "False": False}).astype(object)
            elif kind not in {"string", "unicode", "bytes", "empty"}:
                raise RuntimeError(f"Unsupported object CSV dtype {kind!r}: canonicalize this table before saving.")
            else:
                values = values.astype(object)
            frame[name] = values.mask(nulls, None)
        elif dtype in {"bool", "boolean"}:
            if (~nulls & ~values.isin(["True", "False"])).any():
                raise RuntimeError(f"Invalid saved boolean values: {path.name}/{name}")
            frame[name] = values.map({"True": True, "False": False}).astype(dtype)
        elif dtype == "string":
            frame[name] = values.astype(pd.StringDtype(storage=descriptor.get("string_storage", "python")))
        elif dtype == "category":
            category = pd.CategoricalDtype(descriptor["categories"], ordered=descriptor["ordered"])
            if pd.api.types.infer_dtype(descriptor["categories"], skipna=True) in {"integer", "floating", "mixed-integer-float"}:
                values = pd.to_numeric(values, errors="raise")
            frame[name] = values.astype(category)
        elif dtype.startswith("timedelta64"):
            frame[name] = pd.to_timedelta(values).astype(dtype)
        else:
            frame[name] = values.astype(dtype)
    frame.columns.name = schema.get("column_axis_name")
    return frame

def _p3_read_table(root, descriptor):
    path = _p3_safe_path(root, descriptor["relative_path"])
    schema_path = _p3_safe_path(root, descriptor["schema_path"])
    schema = json.loads(schema_path.read_text(encoding="utf-8"))
    if schema.get("fingerprint") != descriptor["fingerprint"] or schema.get("artifact") != descriptor["relative_path"]:
        raise RuntimeError("Table schema disagrees with the committed catalog.")
    frame = pd.read_parquet(path) if path.suffix == ".parquet" else _p3_restore_csv(path, schema)
    if sector3_table_fingerprint(frame) != descriptor["fingerprint"]:
        raise RuntimeError(f"Typed table readback fingerprint failed: {path.name}")
    return frame

def _p3_catalog_meaning(state):
    return {key: state[key] for key in (
        "schema_version", "complete", "processing_signature", "semantic_config_fingerprint",
        "input_sha256", "core_source_sha256", "config_snapshot", "tables", "summary", "artifacts", "obsolete_paths")}

S4_EXECUTION_METADATA_KEYS = frozenset({
    'run_id', 'run_context', 'run_datetime', 'run_timestamp', 'run_date', 'run_time',
    'timestamp', 'date', 'time', 'execution_date', 'execution_time',
    'created_at', 'created_at_utc', 'created_utc', 'saved_at', 'saved_at_utc', 'saved_utc',
    'started_at', 'started_at_utc', 'started_utc', 'finished_at', 'finished_at_utc', 'finished_utc',
    'generated_at', 'generated_at_utc', 'generated_utc', 'updated_at', 'updated_at_utc', 'updated_utc',
    'modified_at', 'modified_at_utc', 'modified_utc',
})

def sector4_semantic_config(config):
    """Exclude clock bookkeeping only from the effective authored Sector 4 config."""
    def project(value):
        if isinstance(value, dict):
            return {str(key): project(item) for key, item in value.items()
                    if str(key).lower() not in S4_EXECUTION_METADATA_KEYS}
        if isinstance(value, (tuple, list)):
            return [project(item) for item in value]
        return _s3_semantic_json_value(value)
    return project(config)

def sector4_semantic_config_fingerprint(config):
    return _s3_semantic_digest(sector4_semantic_config(config))


In [5]:
def io_s5_stage_sector2(repo, commit, stage_root):
    """Stage the exact published S2 export and verify its current S1 references."""
    import gzip

    def object_from_bytes(payload, label):
        value = json.loads(payload)
        if not isinstance(value, dict):
            raise ValueError(f"{label} must contain a JSON object.")
        return value

    def relative_path(value):
        if (not isinstance(value, str) or not value or "\\" in value or "\x00" in value
                or PurePosixPath(value).is_absolute()
                or any(part in {"", ".", ".."} for part in value.split("/"))
                or not value.endswith((".json", ".csv", ".csv.gz", ".parquet", ".md", ".txt"))):
            raise ValueError(f"Unsafe published S2 metadata path: {value!r}")
        return value

    prefix = "outputs/phase2/sector2/"
    root = Path(stage_root) / "outputs/phase2/sector2"
    catalog_bytes = io_s5_git_blob(repo, commit, prefix + "artifact_catalog.json")
    catalog = object_from_bytes(catalog_bytes, "Published S2 export catalog")
    records = catalog.get("artifacts")
    if catalog.get("schema") != "mmvqa-sector2-export-v1" or not isinstance(records, list) or not records:
        raise ValueError("Publish the supported S2 export catalog to main first.")
    logical_paths, physical_paths, hashes, evidence = set(), set(), {}, {}
    for record in records:
        if not isinstance(record, dict):
            raise ValueError("Published S2 catalog contains a malformed artifact.")
        logical, physical = relative_path(record.get("logical_path")), relative_path(record.get("path"))
        encoding = record.get("encoding")
        if (logical.split("/", 1)[0] not in {"configs", "data", "manifests", "reports", "results"}
                or logical in logical_paths or physical in physical_paths
                or encoding not in {"identity", "gzip"}
                or physical != logical + (".gz" if encoding == "gzip" else "")):
            raise ValueError("Published S2 catalog has duplicate or inconsistent paths/encoding.")
        logical_paths.add(logical)
        physical_paths.add(physical)
        packed = io_s5_git_blob(repo, commit, prefix + physical)
        if len(packed) != record.get("bytes") or hashlib.sha256(packed).hexdigest() != record.get("sha256"):
            raise ValueError(f"Published S2 export checksum failed: {logical}")
        raw = gzip.decompress(packed) if encoding == "gzip" else packed
        digest = hashlib.sha256(raw).hexdigest()
        if logical.startswith(("data/", "manifests/", "results/")) and (
                len(raw) != record.get("source_bytes") or digest != record.get("source_sha256")):
            raise ValueError(f"Published S2 producer-byte checksum failed: {logical}")
        target = root / logical
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(raw)
        hashes[logical] = digest
        evidence[logical] = {"repository_path": prefix + physical, "encoding": encoding,
                             "export_sha256": record["sha256"], "source_sha256": digest}

    required = {"data/sector1_audit_export_report.json", "data/physical_image_inventory.csv",
                "data/metadata_clean.parquet", "manifests/video_manifest.csv",
                "manifests/video_manifest_state.json", "results/decode_audit_state.json",
                "results/frame_alignment_state.json", "results/frame_alignment_attempt_state.json"}
    if not required.issubset(logical_paths):
        raise ValueError(f"Published S2 export lacks required metadata: {sorted(required - logical_paths)}")
    config_relative = relative_path(catalog.get("config_file"))
    if (re.fullmatch(r"configs/phase2_02_video_frame_validation_[0-9_]+_config\.json", config_relative) is None
            or config_relative not in logical_paths):
        raise ValueError("Published S2 catalog must select its exact saved configuration.")
    source2_path = root / config_relative
    source2 = object_from_bytes(source2_path.read_bytes(), "Published S2 configuration")
    state_relatives = {"alignment": "results/frame_alignment_state.json",
                       "decode": "results/decode_audit_state.json",
                       "manifest": "manifests/video_manifest_state.json"}
    state_paths = {name: root / relative for name, relative in state_relatives.items()}
    states = {name: object_from_bytes(path.read_bytes(), f"Published S2 {name} state")
              for name, path in state_paths.items()}
    state_hashes = {name: hashes[relative] for name, relative in state_relatives.items()}
    run_id = source2.get("run_id")
    if not run_id or run_id != catalog.get("run_id") or run_id != states["alignment"].get("run_id"):
        raise ValueError("Published S2 catalog, config and alignment run IDs disagree.")
    attempt_relative = "results/frame_alignment_attempt_state.json"
    attempt = object_from_bytes((root / attempt_relative).read_bytes(), "Published S2 latest attempt")
    if (attempt.get("schema") != "frame-alignment-attempt-v1" or attempt.get("complete") is not True
            or attempt.get("status") != "completed" or not attempt.get("attempt_id")
            or attempt["attempt_id"] != states["alignment"].get("attempt_id")
            or attempt.get("run_id") != run_id
            or attempt.get("final_state_sha256") != state_hashes["alignment"]):
        raise ValueError("The latest published S2 alignment attempt is unfinished or differs from its final state.")

    s1_path = "outputs/phase2/sector1/dataset_audit/audit_export_report.json"
    s1_bytes = io_s5_git_blob(repo, commit, s1_path)
    if s1_bytes != (root / "data/sector1_audit_export_report.json").read_bytes():
        raise ValueError("Published S2 used a different S1 catalog. Rerun/publish S2 from current S1.")
    s1_catalog = object_from_bytes(s1_bytes, "Published S1 catalog")
    for name in ("physical_image_inventory.csv", "metadata_clean.parquet"):
        matching = [item for item in s1_catalog.get("artifacts", [])
                    if isinstance(item, dict) and item.get("filename") == name]
        staged = root / "data" / name
        if (len(matching) != 1 or matching[0].get("file_size_bytes") != staged.stat().st_size
                or matching[0].get("file_sha256") != hashes["data/" + name]):
            raise ValueError(f"The current S1 catalog does not certify published S2 {name}.")
    s1_inventory_path = "outputs/phase2/sector1/dataset_audit/physical_image_inventory.csv"
    if io_s5_git_blob(repo, commit, s1_inventory_path) != (root / "data/physical_image_inventory.csv").read_bytes():
        raise ValueError("Published S2 did not consume the current S1 image inventory.")
    return {"source2": source2, "source2_path": source2_path,
            "dirs2": {name: root / name for name in ("results", "manifests", "configs")},
            "state_paths2": state_paths, "states2": states, "state_hashes2": state_hashes,
            "hashes": hashes, "s1_catalog": s1_catalog,
            "s1_catalog_sha256": hashlib.sha256(s1_bytes).hexdigest(),
            "image_inventory_path": root / "data/physical_image_inventory.csv",
            "attempt": attempt, "attempt_sha256": hashes[attempt_relative],
            "artifact_catalog_sha256": hashlib.sha256(catalog_bytes).hexdigest(),
            "evidence": evidence}


def _s5_validate_reference_chain(context3, context4, states2, state_hashes2, raw2, published2):
    """Verify producer certificates without changing the existing consumed-hash keys."""
    def require(condition, message):
        if not condition:
            raise ValueError(message)

    def digest(value, label):
        require(isinstance(value, str) and re.fullmatch(r"[0-9a-f]{64}", value) is not None,
                f"Missing or malformed {label}; rerun/publish the upstream sector.")
        return value

    def same(actual, expected, label):
        require(actual == expected, f"Published producer reference mismatch: {label}. Rerun/publish the upstream sectors in order.")

    lineage3 = context3["envelope"].get("lineage", {})
    lineage4 = context4["envelope"].get("lineage", {})
    references = lineage3.get("raw_references")
    require(isinstance(references, dict), "S3 lacks validated raw-reference lineage. Rerun/publish S3.")
    require(type(references.get("raw_integrity_version")) is int and references["raw_integrity_version"] == 1,
            "S3 raw-reference contract version is unsupported.")
    for field in ("image_snapshot_id", "video_snapshot_id", "video_state_sha256", "s1_catalog_sha256"):
        digest(references.get(field), "S3 " + field)
    for field, actual in (("sector3_state_sha256", context3["state_sha256"]),
                          ("sector3_config_snapshot_sha256", context3["snapshot_sha256"]),
                          ("sector3_content_id", context3["state"]["content_id"])):
        same(lineage4.get(field), actual, "S4 -> S3 " + field)
    expected_video_reference = {key: references[key] for key in (
        "raw_integrity_version", "video_snapshot_id", "video_state_sha256", "video_count")}
    same(lineage4.get("video_reference"), expected_video_reference, "S4 -> S3 video reference")
    for sector, lineage, key in ((3, lineage3, "source_config_provenance"),
                                  (4, lineage4, "source_config_provenance")):
        source = lineage.get(key, {})
        require(isinstance(source, dict) and source.get("source_branch") == "main"
                and source.get("input_source") == "published_git_objects_only"
                and isinstance(source.get("source_commit"), str)
                and re.fullmatch(r"[0-9a-f]{40}", source["source_commit"]) is not None,
                f"S{sector} must first be rerun and published using its GitHub-main input loader.")
    for name in ("manifest", "alignment", "decode"):
        state = states2[name]
        require(state.get("complete") is True and type(state.get("raw_integrity_version")) is int
                and state["raw_integrity_version"] == 1, f"S2 {name} raw-reference contract is incomplete.")
    for name in ("manifest", "alignment"):
        require(states2[name].get("raw_integrity_complete") is True, f"S2 {name} raw integrity is incomplete.")
    require(states2["decode"].get("all_videos_decoded") is True, "S2 decoding is incomplete.")
    same(references["video_state_sha256"], state_hashes2["manifest"], "S3 -> S2 video state")
    same(references["s1_catalog_sha256"], published2["s1_catalog_sha256"], "S3 -> current S1 catalog")
    same(references["image_snapshot_id"], published2["s1_catalog"].get("image_snapshot_id"), "S3 -> S1 image snapshot")
    same(references["video_snapshot_id"], states2["manifest"].get("video_snapshot_id"), "S3 -> S2 video snapshot")
    for name, expected in (("video_state_sha256", state_hashes2["manifest"]),
                           ("video_manifest_sha256", published2["hashes"]["manifests/video_manifest.csv"]),
                           ("video_snapshot_id", references["video_snapshot_id"])):
        same(states2["decode"].get(name), expected, "S2 decode " + name)
    expected_catalogs = {name: references[name] for name in ("s1_catalog_sha256", "video_state_sha256")}
    same(states2["alignment"].get("raw_reference_catalogs"), expected_catalogs, "S2 alignment catalogs")
    for field, logical in (("audit_export_report.json", "data/sector1_audit_export_report.json"),
                           ("physical_image_inventory.csv", "data/physical_image_inventory.csv"),
                           ("metadata_clean.parquet", "data/metadata_clean.parquet"),
                           ("video_manifest.csv", "manifests/video_manifest.csv"),
                           ("video_manifest_state.json", "manifests/video_manifest_state.json")):
        same(states2["alignment"].get("input_sha256", {}).get(field), published2["hashes"][logical],
             "S2 alignment input " + field)
    for field, expected in expected_catalogs.items():
        same(published2["source2"].get("frame_raw_integrity", {}).get(field), expected, "S2 config " + field)
    for name, state_hash in state_hashes2.items():
        same(lineage3.get("state_sha256", {}).get(name), state_hash, "S3 -> S2 " + name + " state")
    same(lineage3.get("state_sha256", {}).get("alignment_attempt"), published2["attempt_sha256"],
         "S3 -> latest S2 attempt")
    same(lineage3.get("sector2_artifact_catalog_sha256"), published2["artifact_catalog_sha256"], "S3 -> S2 export catalog")
    videos, confirmed = raw2["video_manifest"], raw2["df_alignment_confirmed"]
    for field, expected in (("video_count", len(videos)), ("confirmed_annotation_count", len(confirmed))):
        require(type(references.get(field)) is int, f"S3 {field} must be an integer.")
        same(references[field], expected, "S3 " + field)
    # Compare every confirmed reference to the certified inventories, without media I/O.
    images = pd.read_csv(published2["image_inventory_path"], dtype={"image_relpath": "string", "image_sha256": "string"})
    for table, key, label in ((images, "image_relpath", "S1 images"), (videos, "video_key", "S2 videos")):
        require(table[key].notna().all() and table[key].is_unique, f"{label} needs unique identities.")
    image_index, video_index = images.set_index("image_relpath"), videos.set_index("video_key")
    for target, key, index, source in (
        ("source_image_sha256", "image_relpath", image_index, "image_sha256"),
        ("source_image_size_bytes", "image_relpath", image_index, "image_size_bytes"),
        ("video_relpath", "video_key", video_index, "video_relpath"),
        ("source_video_sha256", "video_key", video_index, "video_sha256"),
        ("source_video_size_bytes", "video_key", video_index, "video_size_bytes"),
    ):
        require(confirmed[target].eq(confirmed[key].map(index[source])).fillna(False).all(),
                f"Confirmed {target} differs from the published source inventory.")
    for field in ("image_snapshot_id", "video_snapshot_id"):
        require(confirmed[field].eq(references[field]).fillna(False).all(),
                f"Confirmed {field} differs from its producing snapshot.")
    return copy.deepcopy(references)


# Pure reference checks: these functions read tables only, never raw media.
S5_SOURCE_RAW_REFERENCE_COLUMNS = (
    "source_image_sha256", "source_image_size_bytes",
    "source_video_sha256", "source_video_size_bytes",
    "image_snapshot_id", "video_snapshot_id",
)
S5_SEGMENT_RAW_REFERENCE_MAP = {
    "target_image_sha256": "source_image_sha256",
    "target_image_size_bytes": "source_image_size_bytes",
    "source_video_sha256": "source_video_sha256",
    "source_video_size_bytes": "source_video_size_bytes",
    "source_video_relpath": "video_relpath",
    "image_snapshot_id": "image_snapshot_id",
    "video_snapshot_id": "video_snapshot_id",
}


def _s5_require_raw_columns(table, columns, name):
    if not isinstance(table, pd.DataFrame) or not table.columns.is_unique:
        raise TypeError(f"{name} must be a DataFrame with unique columns.")
    missing = sorted(set(columns) - set(table.columns))
    if missing:
        raise ValueError(
            f"{name} lacks certified raw references: {missing}. "
            "Rerun and publish Sector 4 from the current certified Sector 3 outputs."
        )


def _s5_validate_raw_values(table, sha_columns, size_columns, path_columns, name):
    for column in sha_columns:
        values = table[column].astype("string")
        if not values.str.fullmatch(r"[0-9a-f]{64}", na=False).all():
            raise ValueError(f"{name}.{column} requires nonempty SHA-256 values.")
    for column in size_columns:
        values = pd.to_numeric(table[column], errors="coerce")
        boolean_values = table[column].map(lambda value: isinstance(value, (bool, np.bool_)))
        if (boolean_values.any() or values.isna().any() or values.le(0).any()
                or values.mod(1).ne(0).any() or not np.isfinite(values.astype("float64")).all()):
            raise ValueError(f"{name}.{column} requires positive integer byte sizes.")
    for column in path_columns:
        for value in table[column]:
            if (not isinstance(value, str) or not value or value != value.strip()
                    or "\\" in value or PurePosixPath(value).is_absolute()
                    or any(part in ("", ".", "..") for part in value.split("/"))):
                raise ValueError(f"{name}.{column} requires canonical relative file paths.")


def validate_segment_raw_references(labelled_manifest):
    """Require consistent identities without equating different JPEG file paths."""
    required = (*S5_SOURCE_RAW_REFERENCE_COLUMNS, "video_key", "image_relpath", "video_relpath")
    _s5_require_raw_columns(labelled_manifest, required, "Sector 5 source")
    _s5_validate_raw_values(
        labelled_manifest,
        ("source_image_sha256", "source_video_sha256", "image_snapshot_id", "video_snapshot_id"),
        ("source_image_size_bytes", "source_video_size_bytes"),
        ("image_relpath", "video_relpath"), "Sector 5 source",
    )
    keys = labelled_manifest["video_key"].astype("string")
    if keys.isna().any() or keys.str.strip().eq("").any():
        raise ValueError("Sector 5 source has a missing video identity.")
    for column in ("image_snapshot_id", "video_snapshot_id"):
        if labelled_manifest[column].nunique(dropna=False) > 1:
            raise ValueError(f"Sector 5 source mixes different {column} references.")
    image_fields = ["source_image_sha256", "source_image_size_bytes", "image_snapshot_id"]
    image_counts = labelled_manifest.groupby("image_relpath", observed=True)[image_fields].nunique(dropna=False)
    if image_counts.gt(1).any(axis=1).any():
        raise ValueError("Annotations of the same source image disagree on its raw reference.")
    video_fields = ["video_relpath", "source_video_sha256", "source_video_size_bytes", "video_snapshot_id"]
    video_counts = labelled_manifest.groupby("video_key", observed=True)[video_fields].nunique(dropna=False)
    if video_counts.gt(1).any(axis=1).any():
        raise ValueError("Annotations of the same source video disagree on its raw reference.")
    if labelled_manifest.groupby("video_relpath", observed=True)["video_key"].nunique(dropna=False).gt(1).any():
        raise ValueError("One source video path is associated with multiple video keys.")


def _s5_validate_target_frame_references(members, targets):
    """One selected physical frame must have one raw image content identity."""
    keys = ["video_key", "opencv_frame_index"]
    selected = targets[keys].drop_duplicates()
    annotations = members.merge(selected, on=keys, how="inner", validate="many_to_one")
    fields = [*S5_SOURCE_RAW_REFERENCE_COLUMNS, "video_relpath"]
    counts = annotations.groupby(keys, observed=True)[fields].nunique(dropna=False)
    if counts.gt(1).any(axis=1).any():
        raise ValueError(
            "Annotations of a representative physical frame disagree on raw image/video references. "
            "Different image paths are allowed only when their SHA-256 and byte size agree."
        )


def validate_segment_manifest_raw_references(segment_manifest, members):
    """Check targets in vectorized tables; check JSON against plain annotation dictionaries."""
    validate_segment_raw_references(members)
    required = (*S5_SEGMENT_RAW_REFERENCE_MAP, "finding_segment_id", "video_key",
                "target_frame_number", "target_opencv_frame_index", "target_image_annotation_id", "target_image_relpath",
                "target_annotations_json", "target_annotation_count")
    _s5_require_raw_columns(segment_manifest, required, "Finding segment manifest")
    _s5_require_raw_columns(members, ("finding_segment_id", "annotation_id", "frame_number", "opencv_frame_index"), "Segment members")
    for column in S5_SEGMENT_RAW_REFERENCE_MAP:
        expected_dtype = "Int64" if column.endswith("_size_bytes") else "string"
        if str(segment_manifest[column].dtype) != expected_dtype:
            raise ValueError(f"Finding segment manifest requires {column} dtype {expected_dtype}.")
    _s5_validate_raw_values(
        segment_manifest,
        ("target_image_sha256", "source_video_sha256", "image_snapshot_id", "video_snapshot_id"),
        ("target_image_size_bytes", "source_video_size_bytes"),
        ("target_image_relpath", "source_video_relpath"), "Finding segment manifest",
    )
    if not segment_manifest["finding_segment_id"].is_unique:
        raise ValueError("Finding segment manifest contains duplicate segment IDs.")
    if set(segment_manifest["finding_segment_id"]) != set(members["finding_segment_id"]):
        raise ValueError("Segment raw-reference validation found inconsistent segment membership.")
    _s5_validate_target_frame_references(members, segment_manifest.rename(
        columns={"target_opencv_frame_index": "opencv_frame_index"}))

    # Match all target annotations once; avoid 22,000+ pandas group/sort/index operations.
    target_keys = segment_manifest[["finding_segment_id", "target_frame_number"]].rename(
        columns={"target_frame_number": "frame_number"})
    target = members.merge(target_keys, on=["finding_segment_id", "frame_number"],
                           how="inner", sort=False, validate="many_to_one")
    if set(target["finding_segment_id"]) != set(segment_manifest["finding_segment_id"]):
        raise ValueError("A segment target has no corresponding source annotation.")
    target = target.sort_values(["finding_segment_id", "annotation_id"], kind="stable")
    selected = segment_manifest.set_index("finding_segment_id")
    first = target.drop_duplicates("finding_segment_id").set_index("finding_segment_id").reindex(selected.index)
    identity_map = {
        "target_image_annotation_id": "annotation_id",
        "target_image_relpath": "image_relpath",
        "target_opencv_frame_index": "opencv_frame_index",
        "video_key": "video_key",
    }
    for target_column, source_column in identity_map.items():
        if not selected[target_column].eq(first[source_column]).fillna(False).all():
            raise ValueError("Segment target differs from its deterministic source annotation.")
    for target_column, source_column in S5_SEGMENT_RAW_REFERENCE_MAP.items():
        if not selected[target_column].eq(first[source_column]).fillna(False).all():
            raise ValueError(f"Segment target raw reference mismatch: {target_column}.")
    target_counts = target.groupby("finding_segment_id", sort=False, observed=True).size().reindex(selected.index)
    if not selected["target_annotation_count"].eq(target_counts).fillna(False).all():
        raise ValueError("Target annotation reference collection has incorrect coverage.")

    annotation_fields = ("image_relpath", "video_relpath", *S5_SOURCE_RAW_REFERENCE_COLUMNS)
    expected_records = {}
    expected_ids = {}
    # All these objects are Python scalars/dictionaries; no pandas access in the JSON loop.
    for record in target[["finding_segment_id", "annotation_id", *annotation_fields]].to_dict(orient="records"):
        annotation_id = record["annotation_id"]
        if annotation_id in expected_records:
            raise ValueError("Target annotation reference collection lost or duplicated annotations.")
        expected_records[annotation_id] = record
        expected_ids.setdefault(record["finding_segment_id"], set()).add(annotation_id)
    for segment_id, encoded in segment_manifest[["finding_segment_id", "target_annotations_json"]].itertuples(
            index=False, name=None):
        try:
            records = json.loads(encoded)
        except (ValueError, TypeError) as error:
            raise ValueError("Target annotation references are not valid JSON.") from error
        ids = expected_ids[segment_id]
        if (not isinstance(records, list) or any(not isinstance(row, dict) for row in records)
                or len(records) != len(ids)):
            raise ValueError("Target annotation reference collection has incorrect coverage.")
        recorded_ids = [row.get("annotation_id") for row in records]
        if (len(set(recorded_ids)) != len(recorded_ids) or set(recorded_ids) != ids):
            raise ValueError("Target annotation reference collection lost or duplicated annotations.")
        for record in records:
            expected = expected_records[record["annotation_id"]]
            for column in annotation_fields:
                if record.get(column) != expected[column]:
                    raise ValueError(f"Target annotation raw reference mismatch: {column}.")



def io_s5_git(repo, *arguments):
    """Run Git without shell expansion, prompts or credential tracing."""
    environment = {key: value for key, value in os.environ.items()
                   if not key.startswith("GIT_TRACE") and key != "GIT_CURL_VERBOSE"}
    environment["GIT_TERMINAL_PROMPT"] = "0"
    process = subprocess.run(["git", *(["-C", str(repo)] if repo is not None else []), *arguments],
                             env=environment, capture_output=True)
    if process.returncode:
        raise RuntimeError("Cannot load published main inputs: "
                           + process.stderr.decode("utf-8", errors="replace").strip())
    return process.stdout


def io_s5_fetch_main(repo_url, repo):
    """Fetch fresh main and pin it once; no local checkout can supply input data."""
    io_s5_git(None, "clone", "--quiet", "--bare", "--filter=blob:none", "--depth=1",
              "--single-branch", "--branch=main", "--", repo_url, str(repo))
    commit = io_s5_git(repo, "rev-parse", "refs/heads/main^{commit}").decode().strip()
    if re.fullmatch(r"[0-9a-f]{40}", commit) is None:
        raise RuntimeError("Published main did not resolve to a full commit SHA.")
    return commit


def _s5_git_relative(value):
    if (not isinstance(value, str) or not value or "\\" in value or "\x00" in value
            or PurePosixPath(value).is_absolute()
            or any(part in {"", ".", ".."} for part in value.split("/"))):
        raise ValueError(f"Unsafe published metadata path: {value!r}")
    return value


def io_s5_git_blob(repo, commit, relative):
    """Read only a regular metadata blob from the immutable selected commit."""
    if re.fullmatch(r"[0-9a-f]{40}", str(commit)) is None:
        raise ValueError("A full pinned main commit SHA is required.")
    relative = _s5_git_relative(relative)
    if (not re.match(r"outputs/phase2/sector[1-4]/", relative)
            or not relative.endswith((".json", ".csv", ".csv.gz", ".parquet", ".md", ".txt"))):
        raise ValueError("S5 accepts only published upstream metadata; raw media is not read.")
    entry = io_s5_git(repo, "ls-tree", "-z", commit, "--", relative)
    if (not entry or entry.count(b"\0") != 1 or b"\t" not in entry
            or entry.split(b"\t", 1)[1] != relative.encode("utf-8") + b"\0"
            or entry.split(b" ", 2)[:2] not in ([b"100644", b"blob"], [b"100755", b"blob"])):
        raise FileNotFoundError(f"Published main {commit[:12]} lacks a regular metadata file: {relative}")
    return io_s5_git(repo, "cat-file", "blob", f"{commit}:{relative}")


def io_s5_stage_catalog(repo, commit, stage_root, sector):
    """Materialize only catalog-declared blobs, then reuse the existing typed reader."""
    prefix = f"outputs/phase2/sector{sector}"
    relative_state = f"configs/sector{sector}_state.json"
    payload = io_s5_git_blob(repo, commit, f"{prefix}/{relative_state}")
    state = json.loads(payload)
    if (not isinstance(state, dict) or state.get("schema_version") != 2
            or state.get("complete") is not True
            or state.get("content_id") != _p3_json_hash(_p3_catalog_meaning(state))):
        raise RuntimeError(f"Publish a complete, current Sector {sector} catalog to main first.")
    records = state["artifacts"]
    paths = [_p3_relative(_s5_git_relative(item["relative_path"])) for item in records]
    if len(paths) != len(set(paths)) or {relative_state, f"configs/sector{sector}_publication_receipt.json"}.intersection(paths):
        raise RuntimeError(f"Sector {sector} catalog contains duplicate or operational entries.")
    root = stage_root / prefix
    target = root / relative_state
    target.parent.mkdir(parents=True)
    target.write_bytes(payload)
    for relative, record in zip(paths, records):
        payload = io_s5_git_blob(repo, commit, f"{prefix}/{relative}")
        if len(payload) != record["bytes"] or hashlib.sha256(payload).hexdigest() != record["sha256"]:
            raise RuntimeError(f"Published Sector {sector} checksum failed: {relative}")
        target = root / relative
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(payload)
    return io_s5_read_catalog(root, sector)


def io_s5_load_published_chain(bootstrap):
    repo_url = bootstrap.get("repo_url")
    if not isinstance(repo_url, str) or re.fullmatch(
            r"https://github\.com/[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", repo_url) is None:
        raise ValueError("S5 repo_url must be a credential-free GitHub HTTPS URL.")
    if bootstrap.get("branch") != "main":
        raise ValueError("S5 inputs must come from published main.")
    for key in ("sector3_dir", "sector4_dir", "sector2_results_dir", "sector2_manifests_dir", "sector2_configs_dir"):
        if bootstrap.get(key) is not None:
            raise ValueError(f"Local input override {key} is unsupported; publish upstream data to main first.")
    workspace = tempfile.TemporaryDirectory(prefix="s5-published-main-")
    try:
        stage_root = Path(workspace.name) / "metadata"
        repo = Path(workspace.name) / "source.git"
        commit = io_s5_fetch_main(repo_url, repo)
        context3 = io_s5_stage_catalog(repo, commit, stage_root, 3)
        context4 = io_s5_stage_catalog(repo, commit, stage_root, 4)
        for sector, context in ((3, context3), (4, context4)):
            github = context["config"].get("github", {})
            if (github.get("branch") != "main"
                    or str(github.get("repo_url", "")).rstrip("/").removesuffix(".git")
                    != repo_url.removesuffix(".git")):
                raise ValueError(f"Published S{sector} configuration points to another repository/branch.")
        published2 = io_s5_stage_sector2(repo, commit, stage_root)
        return {"context3": context3, "context4": context4, "sector2": published2,
                "stage_root": stage_root, "workspace": workspace,
                "provenance": {"repo_url": repo_url, "source_branch": "main", "source_commit": commit,
                               "input_source": "published_git_objects_only"}}
    except BaseException:
        workspace.cleanup()
        raise


def sector5_published_source_identity(lineage):
    """Bind source authority and evidence without invalidation by unrelated main commits."""
    source = lineage.get("published_source", {})
    if (source.get("source_branch") != "main"
            or source.get("input_source") != "published_git_objects_only"
            or re.fullmatch(r"[0-9a-f]{40}", str(source.get("source_commit", ""))) is None):
        raise ValueError("Load the verified published main chain before computing Sector 5.")
    return {key: value for key, value in source.items() if key != "source_commit"}


S5_SECTOR3_INPUTS = (
    "labelled_frame_manifest", "video_frame_manifest",
    "video_class_matrix", "video_class_support",
)
S5_SECTOR4_RESULTS = (
    "labelled_frame_manifest_with_split",
    "video_split_manifest",
    "frame_split_manifest",
)
S5_REQUIRED_LABELLED_COLUMNS = {
    "annotation_id", "annotation_status", "alignment_training_eligible",
    "video_key", "finding_class", "finding_class_normalized",
    "clinical_group", "frame_number", "opencv_frame_index", "frame_index_offset",
    "filename", "image_path", "image_relpath", "has_bbox", "bbox_spatially_usable",
    "bbox_validation_status", "bbox_xmin", "bbox_ymin", "bbox_xmax", "bbox_ymax", "split",
}


def _s5_json(path):
    value = json.loads(Path(path).read_text(encoding="utf-8"))
    if not isinstance(value, dict):
        raise ValueError(f"Expected a JSON object: {path}")
    return value


def _s5_checked_hash(path, expected):
    if not isinstance(expected, str) or re.fullmatch(r"[0-9a-f]{64}", expected) is None:
        raise ValueError(f"Missing or invalid committed SHA-256: {path}")
    actual = _p3_file_hash(path)
    if actual != expected:
        raise RuntimeError(f"Artifact changed after its completed audit: {path}")
    return actual


def io_s5_read_catalog(root, sector):
    """Read one current Sector 3/4 catalog, never choose a config by timestamp."""
    root = Path(root).resolve()
    state_path = _p3_safe_path(root, f"configs/sector{sector}_state.json")
    if not state_path.is_file():
        raise FileNotFoundError(f"Run/save Sector {sector} first; current catalog missing: {state_path}")
    state_hash = _p3_file_hash(state_path)
    state = _s5_json(state_path)
    if state.get("schema_version") != 2 or state.get("complete") is not True:
        raise RuntimeError(f"Sector {sector} catalog is incomplete or has an unsupported schema.")
    if state.get("content_id") != _p3_json_hash(_p3_catalog_meaning(state)):
        raise RuntimeError(f"Sector {sector} catalog content fingerprint is invalid.")
    records = {item["relative_path"]: item for item in state["artifacts"]}
    operational = {f"configs/sector{sector}_state.json", f"configs/sector{sector}_publication_receipt.json"}
    if len(records) != len(state["artifacts"]) or set(records) & operational:
        raise RuntimeError(f"Sector {sector} catalog has duplicate or operational artifact entries.")
    for relative, record in records.items():
        path = _p3_safe_path(root, relative)
        if not path.is_file() or path.stat().st_size != record["bytes"]:
            raise RuntimeError(f"Sector {sector} cataloged file missing or size changed: {relative}")
        _s5_checked_hash(path, record["sha256"])
    if state["config_snapshot"] not in records:
        raise RuntimeError(f"Sector {sector} current config snapshot is absent from its catalog.")
    snapshot_path = _p3_safe_path(root, state["config_snapshot"])
    envelope = _s5_json(snapshot_path)
    semantic = sector3_semantic_config_fingerprint if sector == 3 else sector4_semantic_config_fingerprint
    if (envelope.get("semantic_config_fingerprint") != state["semantic_config_fingerprint"]
            or semantic(envelope["effective_config"]) != state["semantic_config_fingerprint"]):
        raise RuntimeError(f"Sector {sector} effective config disagrees with its catalog.")
    for name, descriptor in state["tables"].items():
        if not {descriptor["relative_path"], descriptor["schema_path"]}.issubset(records):
            raise RuntimeError(f"Sector {sector} table/schema absent from catalog: {name}")
        schema = _s5_json(_p3_safe_path(root, descriptor["schema_path"]))
        if schema.get("artifact") != descriptor["relative_path"] or schema.get("fingerprint") != descriptor["fingerprint"]:
            raise RuntimeError(f"Sector {sector} table schema disagrees with its catalog: {name}")
    summary = state["summary"]
    if not {summary["summary_relative_path"], summary["report_relative_path"]}.issubset(records):
        raise RuntimeError(f"Sector {sector} summary/report absent from catalog.")
    if (_p3_json_hash(_s5_json(_p3_safe_path(root, summary["summary_relative_path"]))) != summary["fingerprint"]
            or records[summary["report_relative_path"]]["sha256"] != summary["report_fingerprint"]):
        raise RuntimeError(f"Sector {sector} summary/report disagrees with its catalog.")
    for relative in state.get("obsolete_paths", []):
        if not _p3_relative(relative).startswith("results/") or relative in records:
            raise RuntimeError(f"Sector {sector} catalog has an invalid obsolete-result entry.")
    return {"root": root, "state": state, "envelope": envelope, "config": envelope["effective_config"],
            "state_path": state_path, "state_sha256": state_hash,
            "snapshot_path": snapshot_path, "snapshot_sha256": _p3_file_hash(snapshot_path)}


def _s5_required_tables(context, names):
    missing = sorted(set(names) - set(context["state"]["tables"]))
    if missing:
        raise KeyError(f"Current catalog at {context['root']} lacks required tables: {missing}")
    return {name: _p3_read_table(context["root"], context["state"]["tables"][name]) for name in names}


def _s5_identity_multiset(table, columns):
    values = table[columns].astype("string")
    for column in columns:
        if column in {"frame_number", "opencv_frame_index", "aligned_frame_index"} or column.startswith("bbox_"):
            values[column] = pd.to_numeric(table[column], errors="coerce").astype("Float64").astype("string")
    return values.fillna("").value_counts(sort=False).sort_index()


def _s5_validate_frame_splits(labelled, split_manifest, frame_split_manifest):
    """Validate authoritative physical-frame assignments and video/split memberships."""
    frame_keys = ["video_key", "opencv_frame_index"]
    pair_keys = ["video_key", "split"]
    allowed = ["train", "validation", "test"]
    for name, table, keys in (
        ("labelled annotations", labelled, frame_keys),
        ("frame_split_manifest", frame_split_manifest, frame_keys),
        ("video_split_manifest", split_manifest, pair_keys),
    ):
        missing = sorted(set(keys + ["split"]) - set(table.columns))
        if missing:
            raise KeyError(f"Sector 4 {name} lacks split fields: {missing}")
        for column in keys + ["split"]:
            values = table[column].astype("string")
            if (values.isna() | values.str.strip().eq("")).any():
                raise ValueError(f"Sector 4 {name} has null/empty {column} values.")
        if not table["split"].isin(allowed).all():
            raise ValueError(f"Sector 4 {name} contains unsupported split names.")
    if frame_split_manifest.duplicated(frame_keys).any():
        raise ValueError("Sector 4 frame_split_manifest requires one row per video/decoded-frame key.")
    if split_manifest.duplicated(pair_keys).any():
        raise ValueError("Sector 4 video_split_manifest requires unique (video_key, split) pairs.")
    indices = frame_split_manifest["opencv_frame_index"]
    numeric = pd.to_numeric(indices, errors="coerce")
    if (numeric.isna() | numeric.lt(0) | numeric.mod(1).ne(0)).any() or (
        not np.isfinite(numeric.astype("float64")).all()
        or indices.map(lambda value: isinstance(value, (bool, np.bool_))).any()
    ):
        raise ValueError("Sector 4 frame_split_manifest requires finite non-negative integer decoded-frame indices.")
    frame_assignments = labelled[frame_keys + ["split"]].drop_duplicates()
    if frame_assignments.duplicated(frame_keys).any():
        raise ValueError("All annotations/classes on one physical frame must share the same split.")
    reference = frame_split_manifest[frame_keys + ["split"]].copy()
    reference["opencv_frame_index"] = numeric.astype("Int64")
    comparison = frame_assignments.merge(
        reference, on=frame_keys, how="outer", validate="one_to_one",
        suffixes=("_annotation", "_manifest"), indicator=True,
    )
    if not comparison["_merge"].eq("both").all() or not comparison[
        "split_annotation"
    ].eq(comparison["split_manifest"]).fillna(False).all():
        raise ValueError("Accepted annotations disagree with the authoritative Sector 4 frame split manifest (frame coverage or split assignment).")
    membership = frame_assignments[pair_keys].drop_duplicates().merge(
        split_manifest[pair_keys], on=pair_keys, how="outer",
        validate="one_to_one", indicator=True,
    )
    if not membership["_merge"].eq("both").all():
        raise ValueError("Sector 4 video/split memberships disagree with the accepted frame assignments.")
    expected_counts = labelled.groupby(pair_keys, observed=True, sort=True).agg(
        retained_annotation_count=("annotation_id", "size"),
        retained_unique_frame_count=("opencv_frame_index", "nunique"),
        retained_class_count=("finding_class_normalized", "nunique"),
    )
    saved_counts = split_manifest.set_index(pair_keys).reindex(expected_counts.index)
    for column in expected_counts:
        if column in saved_counts and not pd.to_numeric(
            saved_counts[column], errors="coerce"
        ).eq(expected_counts[column]).fillna(False).all():
            raise ValueError(f"Sector 4 video/split summary has inconsistent {column} values.")


def _s5_validate_loaded_frames(labelled, sector3_labelled, videos, split_manifest, confirmed, frame_split_manifest):
    validate_segment_raw_references(labelled)
    validate_segment_raw_references(sector3_labelled)
    validate_segment_raw_references(confirmed)
    missing = sorted(S5_REQUIRED_LABELLED_COLUMNS - set(labelled))
    if missing:
        raise KeyError(f"Sector 4 labelled-frame artifact lacks Sector 5 input fields: {missing}")
    if labelled.empty:
        raise ValueError("Sector 5 requires at least one accepted labelled annotation.")
    for column in ("annotation_id", "video_key", "finding_class_normalized", "clinical_group", "split"):
        values = labelled[column].astype("string")
        if (values.isna() | values.str.strip().eq("")).any():
            raise ValueError(f"Sector 5 input has missing/empty {column} values.")
    if not labelled["annotation_id"].is_unique:
        raise ValueError("Sector 4 must preserve unique source annotation_id values.")
    for column in ("frame_number", "opencv_frame_index", "frame_index_offset"):
        values = pd.to_numeric(labelled[column], errors="coerce")
        invalid = values.isna() | ~np.isfinite(values.astype("float64")) | values.mod(1).ne(0)
        if column != "frame_index_offset":
            invalid = invalid | values.lt(0)
        if invalid.any() or labelled[column].map(lambda value: isinstance(value, (bool, np.bool_))).any():
            raise ValueError(f"Sector 5 input requires genuine finite integer {column} values.")
        labelled[column] = values.astype("Int64")
    if not labelled["opencv_frame_index"].eq(labelled["frame_number"] + labelled["frame_index_offset"]).all():
        raise ValueError("Accepted decoded indices disagree with source frame numbers plus verified offsets.")
    if labelled.groupby("video_key", observed=True)["frame_index_offset"].nunique().gt(1).any():
        raise ValueError("Accepted annotations have inconsistent verified offsets within a video.")
    if not labelled["annotation_status"].eq("source_expert_annotation_alignment_confirmed").all():
        raise ValueError("Sector 5 can consume only expert annotations with confirmed alignment.")
    if not labelled["alignment_training_eligible"].eq(True).all():
        raise ValueError("Sector 5 input includes annotations rejected by alignment.")
    if videos["video_key"].isna().any() or not videos["video_key"].is_unique:
        raise ValueError("The completed decoded video inventory needs unique non-null video keys.")
    bounds = labelled["video_key"].map(videos.set_index("video_key")["frame_count"])
    if bounds.isna().any() or not labelled["opencv_frame_index"].lt(bounds).all():
        raise ValueError("Accepted decoded frame indices lie outside audited video bounds.")
    if not videos["frame_count_source"].eq("completed_sector2_decode_audit").all():
        raise ValueError("Video bounds must come from completed Sector 2 decoding.")
    _s5_validate_frame_splits(labelled, split_manifest, frame_split_manifest)
    # Preserve every expert bbox row. Compare annotation identities and multiplicities,
    # and never turn duplicate frame/class rows into new labels or discard them here.
    identity_columns = [column for column in (
        "annotation_id", "filename", "image_key", "video_key", "frame_number",
        "opencv_frame_index", "finding_class_normalized", "bbox_xmin", "bbox_ymin", "bbox_xmax", "bbox_ymax",
        "image_relpath", "video_relpath", "source_image_sha256", "source_image_size_bytes",
        "source_video_sha256", "source_video_size_bytes", "image_snapshot_id", "video_snapshot_id",
    ) if column in sector3_labelled]
    if not _s5_identity_multiset(labelled, identity_columns).equals(_s5_identity_multiset(sector3_labelled, identity_columns)):
        raise ValueError("Sector 4 labelled annotations differ from the current Sector 3 evidence.")
    source_columns = [column for column in (
        "filename", "image_key", "video_key", "frame_number", "finding_class_normalized",
        "bbox_xmin", "bbox_ymin", "bbox_xmax", "bbox_ymax",
        "image_relpath", "video_relpath", "source_image_sha256", "source_image_size_bytes",
        "source_video_sha256", "source_video_size_bytes", "image_snapshot_id", "video_snapshot_id",
    ) if column in confirmed and column in sector3_labelled]
    if not _s5_identity_multiset(confirmed, source_columns).equals(_s5_identity_multiset(sector3_labelled, source_columns)):
        raise ValueError("Sector 3 accepted annotations do not preserve current Sector 2 confirmed evidence.")


def _s5_rebase_live_paths(table, storage_root):
    result = table.copy()
    for path_column, relative_column in (("image_path", "image_relpath"), ("video_path", "video_relpath")):
        if path_column not in result or relative_column not in result:
            continue
        paths = []
        for value in result[relative_column]:
            if pd.isna(value):
                paths.append(pd.NA)
                continue
            relative = PurePosixPath(str(value))
            if relative.is_absolute() or "\\" in str(value) or any(part in {"", ".", ".."} for part in str(value).split("/")):
                raise ValueError(f"Unsafe portable path in {relative_column}: {value!r}")
            paths.append(str(storage_root.joinpath(*relative.parts)))
        result[path_column] = pd.Series(paths, index=result.index).astype(result[path_column].dtype)
    return result


def io_load_sector5_context(bootstrap, settings):
    """Read/verify the current 2 -> 3 -> 4 chain; return config, inputs and lineage."""
    if not isinstance(bootstrap, dict) or not isinstance(settings, dict):
        raise TypeError("BOOTSTRAP and SECTOR5_SETTINGS must be dictionaries.")
    allowed_settings = set(SECTOR5_DEFAULTS) | {"local_output_dir", "export_dir", "push", "dry_run", "github_overrides"}
    unknown = sorted(set(settings) - allowed_settings)
    if unknown:
        raise KeyError(f"Unexpected Sector 5 algorithm settings: {unknown}")
    storage_root = Path(bootstrap["storage_root"]).expanduser().resolve()
    published = io_s5_load_published_chain(bootstrap)
    context3, context4 = published["context3"], published["context4"]
    root3, source4 = context3["root"], context4["config"]
    inputs3 = _s5_required_tables(context3, S5_SECTOR3_INPUTS)
    inputs4 = _s5_required_tables(context4, S5_SECTOR4_RESULTS)
    published2 = published["sector2"]
    source2, source2_path = published2["source2"], published2["source2_path"]
    dirs2 = published2["dirs2"]
    state_paths2 = published2["state_paths2"]
    states2, state_hashes2 = published2["states2"], published2["state_hashes2"]
    run_id = states2["alignment"].get("run_id")
    threshold_keys = {"min_ssim": "frame_alignment_min_ssim", "min_margin": "frame_alignment_min_margin",
                      "candidate_offsets": "frame_index_offset_candidates"}
    if any(states2["alignment"].get("effective_settings", {}).get(recorded) != source2.get(key)
           for recorded, key in threshold_keys.items()):
        raise ValueError("Sector 2 config snapshot disagrees with the completed alignment decision settings.")
    hashes2 = {}
    for family, required in (("alignment", {"frame_alignment_confirmed.parquet", "frame_alignment_excluded.csv"}),
                             ("decode", {"decode_audit.csv"})):
        committed = states2[family].get("output_sha256", {})
        if not required.issubset(committed):
            raise ValueError(f"Sector 2 {family} state lacks required committed output hashes.")
        for filename, digest in committed.items():
            if Path(filename).name != filename or "/" in filename or "\\" in filename:
                raise ValueError("Sector 2 state must list simple committed output filenames.")
            hashes2[filename] = _s5_checked_hash(dirs2["results"] / filename, digest)
    hashes2["video_manifest.csv"] = _s5_checked_hash(dirs2["manifests"] / "video_manifest.csv", states2["manifest"].get("csv_sha256"))
    prior2 = context3["state"]["input_sha256"]
    for filename in ("frame_alignment_confirmed.parquet", "decode_audit.csv", "video_manifest.csv"):
        recorded = {digest for path, digest in prior2.items() if PurePosixPath(str(path)).name == filename}
        if recorded != {hashes2[filename]}:
            raise RuntimeError(f"Current Sector 3 was not built from current Sector 2 {filename}; rerun/save/publish Sector 3 then Sector 4.")
    for name in S5_SECTOR3_INPUTS:
        descriptor = context3["state"]["tables"][name]
        for field in ("relative_path", "schema_path"):
            actual = _p3_file_hash(_p3_safe_path(root3.resolve(), descriptor[field]))
            if context4["state"]["input_sha256"].get(f"{name}/{field}") != actual:
                raise RuntimeError(f"Current Sector 4 was not built from current Sector 3 {name}/{field}; rerun/save/publish Sector 4.")
    raw2 = {"df_alignment_confirmed": pd.read_parquet(dirs2["results"] / "frame_alignment_confirmed.parquet"),
            "frame_alignment_excluded": pd.read_csv(dirs2["results"] / "frame_alignment_excluded.csv", dtype="string"),
            "decode_audit": pd.read_csv(dirs2["results"] / "decode_audit.csv", dtype={"video_key": "string"}),
            "video_manifest": pd.read_csv(dirs2["manifests"] / "video_manifest.csv", dtype={"video_key": "string"})}
    if (len(raw2["df_alignment_confirmed"]) != states2["alignment"].get("confirmed_annotation_rows")
            or len(raw2["frame_alignment_excluded"]) != states2["alignment"].get("excluded_annotation_rows")
            or len(raw2["df_alignment_confirmed"]) + len(raw2["frame_alignment_excluded"]) != states2["alignment"].get("original_annotation_rows")):
        raise ValueError("Sector 2 confirmed/excluded annotation counts disagree with its completed alignment state.")
    if states2["decode"].get("all_videos_decoded") is not True:
        raise ValueError("Every Sector 2 source video must finish decoding before Sector 5.")
    for name, family in (("video_manifest", "manifest"), ("decode_audit", "decode")):
        table = raw2[name]
        if (len(table) != states2[family].get("video_count") or table["video_key"].isna().any()
                or not table["video_key"].is_unique):
            raise ValueError(f"Sector 2 {name} identities/counts disagree with its completed state.")
    if set(raw2["video_manifest"]["video_key"]) != set(raw2["decode_audit"]["video_key"]):
        raise ValueError("Sector 2 video inventory and decode audit contain different video keys.")
    # S2 records certified alignment paths; normalize against its own producer root.
    confirmed = raw2["df_alignment_confirmed"]
    for relative_column, absolute_column in (("image_relpath", "alignment_image_path"),
                                             ("video_relpath", "alignment_video_path")):
        relative_values = confirmed[absolute_column].map(
            lambda value: _s5_git_relative(PurePosixPath(str(value)).relative_to(
                PurePosixPath(source2["storage_root"])).as_posix())).astype("string")
        if relative_column in confirmed and not confirmed[relative_column].eq(relative_values).fillna(False).all():
            raise ValueError(f"S2 alignment path contradicts {relative_column}.")
        confirmed[relative_column] = relative_values
    raw_references = _s5_validate_reference_chain(
        context3, context4, states2, state_hashes2, raw2, published2)
    labelled = inputs4["labelled_frame_manifest_with_split"].copy()
    _s5_validate_loaded_frames(labelled, inputs3["labelled_frame_manifest"], inputs3["video_frame_manifest"],
                              inputs4["video_split_manifest"], raw2["df_alignment_confirmed"],
                              inputs4["frame_split_manifest"])
    config = copy.deepcopy(source4)
    decisions = []
    for key, fallback in SECTOR5_DEFAULTS.items():
        override = settings.get(key)
        value = override if override is not None else source4.get(key, fallback)
        minimum = 1 if key == "segment_max_gap_frames" else 2
        if isinstance(value, (bool, np.bool_)) or not isinstance(value, (int, np.integer)) or value < minimum:
            raise ValueError(f"{key} must be an integer >= {minimum}.")
        config[key] = int(value)
        decisions.append({"config_key": key, "effective_value": int(value),
                          "resolution": "sector5_override" if override is not None else "inherited_sector4" if key in source4 else "declared_sector5_default"})
    sector5_settings = copy.deepcopy(settings)
    sector5_settings.setdefault("local_output_dir", "outputs/phase2/sector5")
    sector5_settings.setdefault("export_dir", "outputs/phase2/sector5")
    if sector5_settings["export_dir"] != "outputs/phase2/sector5":
        raise ValueError("Sector 5 export_dir must be outputs/phase2/sector5.")
    for flag in ("push", "dry_run"):
        if flag in settings and not isinstance(settings[flag], bool):
            raise ValueError(f"SECTOR5_SETTINGS[{flag!r}] must be a boolean.")
    github = copy.deepcopy(source4.get("github", {}))
    github_overrides = copy.deepcopy(settings.get("github_overrides", {}))
    if not isinstance(github_overrides, dict) or set(github_overrides) - {"token_secret_name"}:
        raise ValueError("Sector 5 github_overrides supports only token_secret_name; repository and branch are inherited.")
    if "token_secret_name" in github_overrides:
        name = github_overrides["token_secret_name"]
        if not isinstance(name, str) or not name.strip() or re.search(r"\s", name):
            raise ValueError("token_secret_name must be a nonempty secret reference without whitespace.")
    github.update(github_overrides)
    github.update({"repo_output_dir": sector5_settings["export_dir"],
                   "push": settings.get("push", github.get("push", False)),
                   "dry_run": settings.get("dry_run", github.get("dry_run", False)),
                   "commit_message": "Phase 2 Sector 5: publish finding segments and temporal supervision"})
    github.pop("confirm_push", None)
    config.update({"phase": 2, "sector": 5, "storage_root": str(storage_root),
                   "sector5": sector5_settings, "github": github})
    for key in S4_EXECUTION_METADATA_KEYS:
        config.pop(key, None)
    # Copies used for access are rebased only after validating the saved bytes and identities.
    runtime3 = {name: _s5_rebase_live_paths(table, storage_root) for name, table in inputs3.items()}
    runtime4 = {name: _s5_rebase_live_paths(table, storage_root) for name, table in inputs4.items()}
    runtime4["labelled_frame_manifest_with_split"] = _s5_rebase_live_paths(labelled, storage_root)
    inputs = {"sector2": raw2, "sector3": runtime3, "sector4": runtime4}
    consumed = {f"sector2/{name}": digest for name, digest in hashes2.items()}
    consumed_paths = {f"sector2/{name}": str((dirs2["manifests"] if name == "video_manifest.csv" else dirs2["results"]) / name)
                      for name in hashes2}
    for name, path in state_paths2.items():
        consumed[f"sector2/state/{name}"] = state_hashes2[name]
        consumed_paths[f"sector2/state/{name}"] = str(path)
    if source2_path is not None:
        consumed["sector2/config_snapshot"] = _p3_file_hash(source2_path)
        consumed_paths["sector2/config_snapshot"] = str(source2_path)
    for sector, context, names in ((3, context3, S5_SECTOR3_INPUTS), (4, context4, S5_SECTOR4_RESULTS)):
        for name in names:
            for field in ("relative_path", "schema_path"):
                relative = context["state"]["tables"][name][field]
                consumed[f"sector{sector}/{name}/{field}"] = _p3_file_hash(_p3_safe_path(context["root"], relative))
                consumed_paths[f"sector{sector}/{name}/{field}"] = str(_p3_safe_path(context["root"], relative))
        consumed[f"sector{sector}/state"] = context["state_sha256"]
        consumed_paths[f"sector{sector}/state"] = str(context["state_path"])
        consumed[f"sector{sector}/config_snapshot"] = context["snapshot_sha256"]
        consumed_paths[f"sector{sector}/config_snapshot"] = str(context["snapshot_path"])
    lineage = {"input_sha256": consumed, "input_descriptors": consumed_paths, "sector2_run_id": run_id,
               "sector2_states": states2, "sector2_state_sha256": state_hashes2,
               "sector2_config_snapshot": str(source2_path) if source2_path else None,
               "sector2_config_snapshot_sha256": _p3_file_hash(source2_path) if source2_path else None,
               "upstream": {f"sector{sector}": {"root": str(context["root"]), "content_id": context["state"]["content_id"],
                   "state_sha256": context["state_sha256"], "config_snapshot": str(context["snapshot_path"]),
                   "config_snapshot_sha256": context["snapshot_sha256"]} for sector, context in ((3, context3), (4, context4))},
               "config_inheritance_report": pd.DataFrame.from_records(decisions),
               "config_inheritance_policy": "Published main S2/S3/S4 at one pinned commit; inherit S4 effective config with explicit S5 overrides."}
    for context in (context3, context4):
        _s5_checked_hash(context["state_path"], context["state_sha256"])
    for name, path in state_paths2.items():
        _s5_checked_hash(path, state_hashes2[name])
    for filename, expected in hashes2.items():
        parent = dirs2["manifests"] if filename == "video_manifest.csv" else dirs2["results"]
        _s5_checked_hash(parent / filename, expected)

    lineage["source_table_fingerprint"] = sector3_table_fingerprint(
        runtime4["labelled_frame_manifest_with_split"]
    )

    lineage["raw_references"] = raw_references
    lineage["published_source"] = {
        **published["provenance"],
        "sector2_artifact_catalog_sha256": published2["artifact_catalog_sha256"],
        "s1_catalog_sha256": raw_references["s1_catalog_sha256"],
        "alignment_attempt_sha256": published2["attempt_sha256"],
    }
    lineage["published_inputs"] = {}
    for key, path in consumed_paths.items():
        relative = Path(path).relative_to(published["stage_root"]).as_posix()
        if relative.startswith("outputs/phase2/sector2/"):
            logical = relative.removeprefix("outputs/phase2/sector2/")
            evidence = copy.deepcopy(published2["evidence"][logical])
        else:
            evidence = {"repository_path": relative, "encoding": "identity",
                        "export_sha256": consumed[key], "source_sha256": consumed[key]}
        lineage["published_inputs"][key] = evidence
    # Keep this private workspace alive for the existing pre-save checksum gate.
    # Only config/lineage/tables are persisted, never this operational handle.
    return {"config": config, "inputs": inputs, "lineage": lineage,
            "_source_workspace": published["workspace"]}


### 4. Load published metadata, then mount output storage

Preserve S4 frame-level split assignments. Check that S4 consumed the current published S3,
and S3 consumed the current published S2 and S1 references. Full commit provenance is retained
in lineage. Cache identity follows certified content; unrelated main commits do not force changes.


In [6]:
# Drop stale runtime results before a new source load; failures cannot reuse them.
for _s5_runtime_name in (
    "SECTOR5_CONTEXT", "CONFIG", "INPUTS", "LINEAGE", "SOURCE_LABELLED_FRAME_MANIFEST",
    "SECTOR5_RESULT", "SECTOR5_TABLES", "SECTOR5_PERSISTED", "SECTOR5_PUBLICATION",
    "PROCESSING_SIGNATURE", "labelled_frame_manifest", "finding_segment_manifest",
):
    globals().pop(_s5_runtime_name, None)

SECTOR5_CONTEXT = io_load_sector5_context(BOOTSTRAP, SECTOR5_SETTINGS)
CONFIG = SECTOR5_CONTEXT["config"]
INPUTS = SECTOR5_CONTEXT["inputs"]
LINEAGE = SECTOR5_CONTEXT["lineage"]
print("Pinned published main commit:", LINEAGE["published_source"]["source_commit"])

if BOOTSTRAP.get("mount_google_drive", False) and str(BOOTSTRAP["storage_root"]).startswith("/content/drive/"):
    try:
        from google.colab import drive
    except ImportError:
        if not Path(BOOTSTRAP["storage_root"]).is_dir():
            raise RuntimeError("This storage root is on Google Drive. Open the notebook in Colab, or set BOOTSTRAP['storage_root'] to a mounted local copy.") from None
    else:
        drive.mount("/content/drive", force_remount=False)


SECTOR2_INPUTS = INPUTS["sector2"]
SECTOR3_INPUTS = INPUTS["sector3"]
SECTOR4_INPUTS = INPUTS["sector4"]

df_alignment_confirmed = SECTOR2_INPUTS["df_alignment_confirmed"]
frame_alignment_excluded = SECTOR2_INPUTS["frame_alignment_excluded"]
decode_audit = SECTOR2_INPUTS["decode_audit"]
video_frame_manifest = SECTOR3_INPUTS["video_frame_manifest"]
video_class_matrix = SECTOR3_INPUTS["video_class_matrix"]
video_class_support = SECTOR3_INPUTS["video_class_support"]
labelled_frame_manifest = SECTOR4_INPUTS["labelled_frame_manifest_with_split"].copy()
video_split_manifest = SECTOR4_INPUTS["video_split_manifest"]
frame_split_manifest = SECTOR4_INPUTS["frame_split_manifest"]

print("Verified upstream chain: Sector 2 -> Sector 3 -> Sector 4")
print("Accepted annotation rows:", f"{len(labelled_frame_manifest):,}")
print("Unique verified frame/class anchors:", f"{len(labelled_frame_manifest.drop_duplicates(['video_key', 'opencv_frame_index', 'finding_class_normalized'])):,}")
display(LINEAGE["config_inheritance_report"])


Pinned published main commit: 6ae4990846104b2f7592256170f550f3b1c2b4ff
Mounted at /content/drive
Verified upstream chain: Sector 2 -> Sector 3 -> Sector 4
Accepted annotation rows: 47,239
Unique verified frame/class anchors: 47,238


,config_key,effective_value,resolution
0,segment_max_gap_frames,1,declared_sector5_default
1,minimum_verified_segment_frames,2,declared_sector5_default


### 5. Annotation preserving input

One input row is one source annotation, identified by annotation_id.
A frame may have several classes, or several annotations/boxes of the same
class. Both cases are supported; annotations are never removed to build segments.
A temporal anchor is the distinct (`video_key`, `frame_number`, `finding_class_normalized`) combination. Multiple annotations on that anchor
receive the same segment ID and contribute one verified frame.
For example, two lesion annotations on frame 100 and one on frame 101 produce
one segment with 2 verified frames and 3 annotations when the gap threshold
is at least 1. A different class on frame 100 belongs to its own class segment.
Duplicate annotation IDs and inconsistent decoded-frame indices remain errors.
Different source image paths and raw label spellings are preserved. All annotations of a representative physical frame must agree on image content, video identity and snapshot references.

In [7]:
required = {
    "annotation_id",
    "video_key", "finding_class", "finding_class_normalized", "clinical_group",
    "frame_number", "opencv_frame_index", "filename", "image_path", "image_relpath",
    "has_bbox", "bbox_spatially_usable", "bbox_validation_status",
    "bbox_xmin", "bbox_ymin", "bbox_xmax", "bbox_ymax", "split",
}
missing = sorted(required - set(labelled_frame_manifest.columns))
if missing:
    raise KeyError(f"Sector 5 input is missing required columns: {missing}")
for column in ("video_key", "finding_class", "finding_class_normalized", "clinical_group", "split"):
    values = labelled_frame_manifest[column].astype("string").str.strip()
    if values.isna().any() or values.eq("").any():
        raise ValueError(f"Sector 5 input contains missing/empty {column} values.")
if not labelled_frame_manifest["split"].isin(["train", "validation", "test"]).all():
    raise ValueError("Sector 5 requires the train/validation/test names from Sector 4.")
if labelled_frame_manifest.groupby(
    ["video_key", "opencv_frame_index"], observed=True
)["split"].nunique(dropna=False).gt(1).any():
    raise ValueError("One physical frame cannot appear in more than one supervised split.")
for column in ("frame_number", "opencv_frame_index"):
    values = pd.to_numeric(labelled_frame_manifest[column], errors="coerce")
    invalid = values.isna() | values.lt(0) | values.mod(1).ne(0)
    if not np.isfinite(values.astype("float64")).all() or invalid.any():
        raise ValueError(f"Sector 5 requires finite non-negative integer {column} values.")
    labelled_frame_manifest[column] = values.astype("Int64")
annotation_ids = labelled_frame_manifest["annotation_id"].astype("string").str.strip()
if annotation_ids.isna().any() or annotation_ids.eq("").any() or annotation_ids.duplicated().any():
    raise ValueError("Every source annotation must have a non-empty, unique annotation_id.")
physical_frame_columns = ["opencv_frame_index"]
physical_frame_counts = labelled_frame_manifest.groupby(
    ["video_key", "frame_number"], observed=True)[physical_frame_columns].nunique(dropna=False)
if physical_frame_counts.gt(1).any(axis=1).any():
    raise ValueError("Annotations on the same video/frame disagree on aligned frame index.")
anchor_columns = ["video_key", "frame_number", "finding_class_normalized"]
anchor_count = len(labelled_frame_manifest[anchor_columns].drop_duplicates())
multiple_annotation_mask = labelled_frame_manifest.duplicated(anchor_columns, keep=False)
if multiple_annotation_mask.any():
    details = anchor_columns + ["annotation_id", "bbox_xmin", "bbox_ymin", "bbox_xmax", "bbox_ymax"]
    print("Multiple annotations on the same frame/class are retained:")
    display(labelled_frame_manifest.loc[multiple_annotation_mask, details].head(30))
print("Sector 5 input ready:", len(labelled_frame_manifest), "source annotations;",
      anchor_count, "distinct frame/class anchors")

source_candidate = labelled_frame_manifest.copy(deep=True)

if (
    sector3_table_fingerprint(source_candidate)
    != LINEAGE.get("source_table_fingerprint")
):
    raise RuntimeError(
        "Sector 5 input changed after verified loading. "
        "Rerun Sections 4 and 5."
    )

SOURCE_LABELLED_FRAME_MANIFEST = source_candidate
del source_candidate

Multiple annotations on the same frame/class are retained:


,video_key,frame_number,finding_class_normalized,annotation_id,bbox_xmin,bbox_ymin,bbox_xmax,bbox_ymax
41840,eb0203196e284797,2380,erosion,annotation_66a9845407f9b4e6_0,1.0,190.0,61.0,244.0
41841,eb0203196e284797,2380,erosion,annotation_fea57071cd752d01_0,208.0,178.0,260.0,226.0


Sector 5 input ready: 47239 source annotations; 47238 distinct frame/class anchors


### 6. Build verified finding segments

Define the segmentation function here; execution takes place once in Section 11.
A new segment starts at the first anchor, a gap larger than `segment_max_gap_frames`, or an observed split-run change.
Split runs are determined across all accepted physical frames of each video before grouping by finding class. Unknown intermediate frames remain unknown.


In [8]:
def add_verified_segment_numbers(labelled_manifest, config):
    """Preserve annotations; split segments at gaps or observed split-run boundaries."""
    group_columns = ["video_key", "finding_class_normalized"]
    anchor_columns = group_columns + ["frame_number"]
    missing_columns = sorted(set(anchor_columns + ["split"]) - set(labelled_manifest.columns))
    if missing_columns:
        raise KeyError(f"Cannot build verified segments. Missing columns: {missing_columns}")
    max_gap = config["segment_max_gap_frames"]
    if isinstance(max_gap, bool) or not isinstance(max_gap, (int, np.integer)) or max_gap < 1:
        raise ValueError("segment_max_gap_frames must be a positive integer.")
    if not labelled_manifest["split"].isin(["train", "validation", "test"]).all():
        raise ValueError("Every segment anchor requires a valid Sector 4 split.")
    physical_keys = ["video_key", "frame_number"]
    frames = labelled_manifest[physical_keys + ["split"]].drop_duplicates()
    if frames.duplicated(physical_keys).any():
        raise ValueError("Annotations/classes on one physical frame have conflicting splits.")
    frames = frames.sort_values(physical_keys, kind="stable").reset_index(drop=True)
    previous_split = frames.groupby("video_key", sort=False, observed=True)["split"].shift()
    new_run = previous_split.isna() | frames["split"].ne(previous_split)
    frames["_s5_split_run"] = new_run.astype("int64").groupby(
        frames["video_key"], sort=False, observed=True).cumsum()
    sort_columns = anchor_columns + (["annotation_id"] if "annotation_id" in labelled_manifest else [])
    result = labelled_manifest.sort_values(sort_columns, kind="stable").reset_index(drop=True).copy()
    anchors = result[anchor_columns].drop_duplicates().merge(
        frames[physical_keys + ["_s5_split_run"]], on=physical_keys,
        how="left", sort=False, validate="many_to_one")
    groups = anchors.groupby(group_columns, sort=False, observed=True)
    frame_gap = groups["frame_number"].diff()
    previous_run = groups["_s5_split_run"].shift()
    starts_new_segment = (frame_gap.isna() | frame_gap.gt(max_gap)
                          | anchors["_s5_split_run"].ne(previous_run))
    # Number across all runs of each video/class; preserve the existing ID format
    # without reusing the same ordinal separately in train/validation/test.
    anchors["segment_number"] = starts_new_segment.astype("int64").groupby(
        [anchors["video_key"], anchors["finding_class_normalized"]],
        sort=False, observed=True).cumsum().astype("Int64")
    return result.drop(columns="segment_number", errors="ignore").merge(
        anchors[anchor_columns + ["segment_number"]], on=anchor_columns,
        how="left", sort=False, validate="many_to_one")


### 7. Create finding segment IDs

The supplied ID format is `video__class__00001`. It is reproducible for the same
inputs, gap policy, and frame-split assignments. It is an **ordinal identity**: adding/removing an earlier
segment or changing grouping may renumber later IDs. Normalization collisions
are checked explicitly. This is not a content-hash identifier.

In [9]:
def normalize_identifier_series(values):
    """Converts a Series into stable identifier components."""
    return (values.astype("string").str.strip().str.casefold()
            .str.replace(r"[^a-z0-9]+", "_", regex=True).str.strip("_"))

def add_finding_segment_ids(labelled_manifest):
    """Creates reproducible ordinal IDs for a fixed segment inventory."""
    identity_columns = ["video_key", "finding_class_normalized", "segment_number"]
    missing_columns = sorted(set(identity_columns) - set(labelled_manifest.columns))
    if missing_columns:
        raise KeyError("Cannot create finding segment IDs. "
                       f"Missing columns: {missing_columns}")
    result = labelled_manifest.copy()
    video_ids = normalize_identifier_series(result["video_key"])
    class_ids = normalize_identifier_series(result["finding_class_normalized"])
    segment_numbers = pd.to_numeric(result["segment_number"], errors="coerce")
    invalid_video_ids = video_ids.isna() | video_ids.eq("")
    invalid_class_ids = class_ids.isna() | class_ids.eq("")
    invalid_segment_numbers = (segment_numbers.isna() | segment_numbers.lt(1)
                               | segment_numbers.mod(1).ne(0))
    if invalid_video_ids.any():
        raise ValueError("Cannot create finding segment IDs: "
                         f"{int(invalid_video_ids.sum())} rows have invalid video keys.")
    if invalid_class_ids.any():
        raise ValueError("Cannot create finding segment IDs: "
                         f"{int(invalid_class_ids.sum())} rows have invalid finding classes.")
    if invalid_segment_numbers.any():
        raise ValueError("Cannot create finding segment IDs: "
                         f"{int(invalid_segment_numbers.sum())} rows have invalid segment numbers.")
    segment_number_ids = (segment_numbers.astype("Int64").astype("string")
                          .str.zfill(5))
    finding_segment_ids = (video_ids.str.cat(class_ids, sep="__")
                          .str.cat(segment_number_ids, sep="__"))
    result = result.assign(segment_number=segment_numbers.astype("Int64"),
                           finding_segment_id=finding_segment_ids)
    segment_identity = result[identity_columns + ["finding_segment_id"]].drop_duplicates()
    collision_mask = segment_identity["finding_segment_id"].duplicated(keep=False)
    if collision_mask.any():
        collisions = (segment_identity.loc[collision_mask]
                      .sort_values("finding_segment_id").to_dict(orient="records"))
        raise RuntimeError(f"Finding segment ID collisions detected: {collisions}")
    return result


### 8. Build the finding-segment manifest

One row represents one finding segment. `verified_frame_count` counts distinct
frame/class anchors under the input contract; it does not count all frames in the
inclusive span. The target is the middle verified frame in temporal order, with
the **upper middle** chosen for even counts. Its image and bbox fields are copied
from that anchor, without preferring a bbox or inspecting image quality.

The manifest carries the certified target-image hash and byte size, source-video
hash, byte size and portable path, and both upstream snapshot IDs. Every
annotation of a selected physical frame must agree on these raw identities;
different image paths are allowed when their bytes have the same reference.
The deterministic first annotation supplies the target path, while the JSON
collection preserves each annotation and its own source references. These
checks use metadata only and do not open JPEG or video files.


In [10]:
SEGMENT_MANIFEST_COLUMNS = [
    "finding_segment_id", "video_key", "finding_class", "finding_class_normalized",
    "finding_class_variants_json",
    "clinical_group", "verified_start_frame", "verified_end_frame",
    "verified_frame_count", "verified_annotation_count",
    "target_frame_number", "target_opencv_frame_index", "target_filename",
    "target_image_path", "target_image_relpath", "target_image_sha256", "target_image_size_bytes",
    "source_video_sha256", "source_video_size_bytes", "source_video_relpath",
    "image_snapshot_id", "video_snapshot_id", "target_image_annotation_id", "target_annotation_count",
    "target_annotation_ids_json", "target_annotations_json", "target_bbox_mode",
    "target_bbox_count", "target_usable_bbox_count",
    "target_has_bbox", "target_bbox_spatially_usable", "target_bbox_validation_status",
    "target_bbox_xmin", "target_bbox_ymin", "target_bbox_xmax", "target_bbox_ymax", "split",
]

def _segment_annotation_json(rows):
    """Keep all target annotation metadata in a portable JSON scalar string."""
    columns = ["annotation_id", "finding_class", "finding_class_normalized",
               "frame_number", "opencv_frame_index", "filename", "image_path", "image_relpath",
               "video_relpath", *S5_SOURCE_RAW_REFERENCE_COLUMNS,
               "has_bbox", "bbox_spatially_usable", "bbox_validation_status",
               "bbox_xmin", "bbox_ymin", "bbox_xmax", "bbox_ymax"]
    records = []
    for record in rows[columns].to_dict(orient="records"):
        records.append({key: None if pd.isna(value) else
                        value.item() if isinstance(value, np.generic) else value
                        for key, value in record.items()})
    return json.dumps(records, ensure_ascii=False, allow_nan=False, separators=(",", ":"))

def build_finding_segment_manifest(labelled_manifest):
    """Summarize unique frames and preserve every representative-frame annotation."""
    required_columns = {
        "annotation_id", "finding_segment_id", "video_key", "finding_class",
        "finding_class_normalized", "clinical_group", "frame_number", "opencv_frame_index",
        "filename", "image_path", "image_relpath", "has_bbox", "bbox_spatially_usable",
        "bbox_validation_status", "bbox_xmin", "bbox_ymin", "bbox_xmax", "bbox_ymax", "split",
    }
    missing_columns = sorted(required_columns - set(labelled_manifest.columns))
    if missing_columns:
        raise KeyError("Cannot build finding segment manifest. "
                       f"Missing columns: {missing_columns}")
    validate_segment_raw_references(labelled_manifest)
    if labelled_manifest.empty:
        empty = pd.DataFrame(columns=SEGMENT_MANIFEST_COLUMNS)
        for column in S5_SEGMENT_RAW_REFERENCE_MAP:
            empty[column] = pd.Series(dtype="Int64" if column.endswith("_size_bytes") else "string")
        return empty
    identifiers = labelled_manifest["finding_segment_id"].astype("string").str.strip()
    if (identifiers.isna() | identifiers.eq("")).any():
        raise ValueError("Finding segment manifest contains rows without a valid segment ID.")
    annotation_ids = labelled_manifest["annotation_id"].astype("string").str.strip()
    if annotation_ids.isna().any() or annotation_ids.eq("").any() or annotation_ids.duplicated().any():
        raise ValueError("Source annotation IDs must be non-empty and unique.")
    result = labelled_manifest.sort_values(
        ["finding_segment_id", "frame_number", "annotation_id"], kind="stable").reset_index(drop=True)
    invariant_columns = ["video_key", "finding_class_normalized",
                         "clinical_group", "split"]
    segment_group = result.groupby("finding_segment_id", sort=False, observed=True)
    inconsistent = segment_group[invariant_columns].nunique(dropna=False).gt(1).any(axis=1)
    if inconsistent.any():
        raise ValueError("Inconsistent metadata found within finding segments: "
                         f"{inconsistent.index[inconsistent].tolist()}")
    image_columns = ["opencv_frame_index", "filename", "image_path", "image_relpath"]
    image_counts = result.groupby(["video_key", "frame_number"], sort=False,
                                  observed=True)[["opencv_frame_index"]].nunique(dropna=False)
    if image_counts.gt(1).any(axis=1).any():
        raise ValueError("Annotations on the same video/frame disagree on aligned frame index.")
    segment_summary = segment_group.agg(
        video_key=("video_key", "first"), finding_class=("finding_class", lambda values: sorted(set(values))[0]),
        finding_class_normalized=("finding_class_normalized", "first"),
        finding_class_variants_json=("finding_class", lambda values: json.dumps(sorted(set(values)), ensure_ascii=False)),
        clinical_group=("clinical_group", "first"),
        verified_start_frame=("frame_number", "min"), verified_end_frame=("frame_number", "max"),
        verified_frame_count=("frame_number", "nunique"),
        verified_annotation_count=("annotation_id", "size"), split=("split", "first"),
    ).reset_index()
    # This is only an anchor view; the original annotation table stays complete.
    anchors = result.drop_duplicates(["finding_segment_id", "frame_number"])
    anchor_group = anchors.groupby("finding_segment_id", sort=False, observed=True)
    target_mask = anchor_group.cumcount().eq(
        anchor_group["frame_number"].transform("size").floordiv(2))
    target_columns = ["frame_number"] + image_columns
    target_frames = anchors.loc[target_mask, ["finding_segment_id"] + target_columns]
    target_annotations = result.merge(
        target_frames[["finding_segment_id", "frame_number"]],
        on=["finding_segment_id", "frame_number"], how="inner", sort=False,
        validate="many_to_one")
    # Check every annotation of each selected physical frame, including other classes.
    _s5_validate_target_frame_references(result, target_annotations)
    bbox_columns = ["has_bbox", "bbox_spatially_usable", "bbox_validation_status",
                   "bbox_xmin", "bbox_ymin", "bbox_xmax", "bbox_ymax"]
    target_rows = []
    for segment_id, rows in target_annotations.groupby("finding_segment_id", sort=False, observed=True):
        first = rows.iloc[0]
        single_annotation = len(rows) == 1
        record = {"finding_segment_id": segment_id,
                  **{"target_" + column: first[column] for column in target_columns},
                  **{target: first[source] for target, source in S5_SEGMENT_RAW_REFERENCE_MAP.items()},
                  "target_image_annotation_id": first["annotation_id"],
                  "target_annotation_count": len(rows),
                  "target_annotation_ids_json": json.dumps(rows["annotation_id"].tolist(), ensure_ascii=False),
                  "target_annotations_json": _segment_annotation_json(rows),
                  "target_bbox_mode": "single_annotation" if single_annotation else "annotation_collection",
                  "target_bbox_count": int(rows["has_bbox"].fillna(False).sum()),
                  "target_usable_bbox_count": int(rows["bbox_spatially_usable"].fillna(False).sum())}
        if single_annotation:
            record.update({"target_" + column: first[column] for column in bbox_columns})
        else:
            record.update(target_has_bbox=bool(rows["has_bbox"].fillna(False).any()),
                          target_bbox_spatially_usable=False,
                          target_bbox_validation_status="multiple_annotations_use_member_manifest",
                          **{"target_" + column: pd.NA for column in bbox_columns[3:]})
        target_rows.append(record)
    manifest = segment_summary.merge(pd.DataFrame(target_rows), on="finding_segment_id",
                                     how="left", validate="one_to_one")
    for column in ["verified_start_frame", "verified_end_frame", "verified_frame_count",
                   "verified_annotation_count", "target_frame_number", "target_opencv_frame_index",
                   "target_annotation_count", "target_bbox_count", "target_usable_bbox_count",
                   "target_image_size_bytes", "source_video_size_bytes"]:
        manifest[column] = manifest[column].astype("Int64")
    for column in ["target_has_bbox", "target_bbox_spatially_usable"]:
        manifest[column] = manifest[column].astype("boolean")
    for column in ["target_bbox_xmin", "target_bbox_ymin", "target_bbox_xmax", "target_bbox_ymax"]:
        manifest[column] = manifest[column].astype("Float64")
    for column in ["finding_class_variants_json", "target_image_annotation_id",
                   "target_annotation_ids_json", "target_annotations_json", "target_bbox_mode",
                   "target_bbox_validation_status", "target_image_sha256", "source_video_sha256",
                   "source_video_relpath", "image_snapshot_id", "video_snapshot_id"]:
        manifest[column] = manifest[column].astype("string")
    return manifest.loc[:, SEGMENT_MANIFEST_COLUMNS].sort_values(
        ["video_key", "verified_start_frame", "finding_class_normalized"],
        kind="stable").reset_index(drop=True)

### 9. Split/class distribution report

The report retains zero-support combinations for classes present anywhere in the
accepted input taxonomy. Frame, source-video and segment supports are counted
independently. With multilabel images, `frame_fraction_within_split` is the share
of **class/frame supports**, not the share of unique physical images.

In [11]:
SPLIT_CLASS_DISTRIBUTION_COLUMNS = [
    "split", "finding_class_normalized", "clinical_group", "labelled_frame_count",
    "video_count", "segment_count", "frame_fraction_within_split", "class_present_in_split",
]

def build_split_class_distribution_report(labelled_manifest, segment_manifest):
    """Counts verified frames, source videos and segments independently per split/class."""
    required_labelled_columns = {"split", "video_key", "opencv_frame_index",
                                "finding_class_normalized", "clinical_group"}
    required_segment_columns = {"finding_segment_id", "split", "video_key",
                               "finding_class_normalized"}
    missing_labelled_columns = sorted(required_labelled_columns - set(labelled_manifest.columns))
    missing_segment_columns = sorted(required_segment_columns - set(segment_manifest.columns))
    if missing_labelled_columns:
        raise KeyError("Split-class report is missing labelled-frame "
                       f"columns: {missing_labelled_columns}")
    if missing_segment_columns:
        raise KeyError("Split-class report is missing segment columns: "
                       f"{missing_segment_columns}")
    expected_splits = pd.DataFrame({"split": ["train", "validation", "test"]})
    actual_splits = set(labelled_manifest["split"].dropna().astype("string").unique())
    unexpected_splits = sorted(actual_splits - set(expected_splits["split"]))
    if unexpected_splits:
        raise ValueError(f"Unexpected supervised split names: {unexpected_splits}")
    taxonomy = (labelled_manifest[["finding_class_normalized", "clinical_group"]]
                .drop_duplicates().sort_values(
                    ["clinical_group", "finding_class_normalized"], kind="stable")
                .reset_index(drop=True))
    inconsistent_taxonomy_mask = taxonomy["finding_class_normalized"].duplicated(keep=False)
    if inconsistent_taxonomy_mask.any():
        inconsistent_classes = taxonomy.loc[inconsistent_taxonomy_mask].to_dict(orient="records")
        raise ValueError("Finding classes map to inconsistent clinical "
                         f"groups: {inconsistent_classes}")
    split_class_grid = expected_splits.merge(taxonomy, how="cross")
    frame_support = (labelled_manifest[
        ["split", "finding_class_normalized", "video_key", "opencv_frame_index"]]
        .drop_duplicates().groupby(["split", "finding_class_normalized"],
            as_index=False, sort=False, observed=True).agg(
                labelled_frame_count=("opencv_frame_index", "size"),
                video_count=("video_key", "nunique")))
    segment_support = (segment_manifest[
        ["finding_segment_id", "split", "finding_class_normalized"]]
        .drop_duplicates(subset=["finding_segment_id"])
        .groupby(["split", "finding_class_normalized"],
                 as_index=False, sort=False, observed=True)
        .agg(segment_count=("finding_segment_id", "nunique")))
    count_columns = ["labelled_frame_count", "video_count", "segment_count"]
    return (split_class_grid.merge(frame_support,
                on=["split", "finding_class_normalized"], how="left", validate="one_to_one")
        .merge(segment_support, on=["split", "finding_class_normalized"],
               how="left", validate="one_to_one")
        .assign(**{column: lambda data, column=column:
                   data[column].fillna(0).astype("Int64") for column in count_columns})
        .assign(frame_fraction_within_split=lambda data:
            data["labelled_frame_count"].astype("float64")
            .div(data.groupby("split", observed=True)["labelled_frame_count"]
                 .transform("sum").astype("float64")).fillna(0.0),
            class_present_in_split=lambda data: data["labelled_frame_count"].gt(0))
        .loc[:, SPLIT_CLASS_DISTRIBUTION_COLUMNS]
        .sort_values(["split", "labelled_frame_count", "finding_class_normalized"],
                     ascending=[True, False, True], kind="stable")
        .reset_index(drop=True))


### 10. Determine temporal-grounding supervision

- `single_frame_anchor_only`: one verified anchor.
- `multi_frame_weak_supervision`: at least the configured number of anchors and a
  nonzero observed frame span; this marks a temporal-grounding **candidate**.
- `insufficient_multi_frame_supervision`: multiple anchors below the configured
  threshold, or without a temporal extent.

`verified_temporal_boundary_available` remains false. Passing the threshold does
not prove continuous visibility or exact clinical onset/offset.

In [12]:
def add_temporal_supervision_metadata(segment_manifest, config):
    """Observed segment limits are not exact clinical onset/offset boundaries."""
    required_columns = {"verified_start_frame", "verified_end_frame", "verified_frame_count"}
    missing_columns = sorted(required_columns - set(segment_manifest.columns))
    if missing_columns:
        raise KeyError("Cannot add temporal supervision metadata. "
                       f"Missing columns: {missing_columns}")
    minimum_frames = config["minimum_verified_segment_frames"]
    if (isinstance(minimum_frames, bool)
            or not isinstance(minimum_frames, (int, np.integer)) or minimum_frames < 2):
        raise ValueError("minimum_verified_segment_frames must be "
                         "an integer greater than or equal to 2.")
    verified_frame_counts = pd.to_numeric(segment_manifest["verified_frame_count"], errors="coerce")
    verified_start_frames = pd.to_numeric(segment_manifest["verified_start_frame"], errors="coerce")
    verified_end_frames = pd.to_numeric(segment_manifest["verified_end_frame"], errors="coerce")
    invalid_count_mask = (verified_frame_counts.isna() | verified_frame_counts.lt(1)
                          | verified_frame_counts.mod(1).ne(0))
    invalid_range_mask = (verified_start_frames.isna() | verified_end_frames.isna()
                          | verified_start_frames.lt(0)
                          | verified_end_frames.lt(verified_start_frames))
    if invalid_count_mask.any():
        raise ValueError(f"Invalid verified frame counts found: {int(invalid_count_mask.sum())}")
    if invalid_range_mask.any():
        raise ValueError(f"Invalid verified temporal ranges found: {int(invalid_range_mask.sum())}")
    has_temporal_extent = verified_end_frames.gt(verified_start_frames)
    temporal_grounding_candidate = verified_frame_counts.ge(minimum_frames) & has_temporal_extent
    temporal_supervision_level = np.select(
        [temporal_grounding_candidate, verified_frame_counts.eq(1)],
        ["multi_frame_weak_supervision", "single_frame_anchor_only"],
        default="insufficient_multi_frame_supervision")
    return segment_manifest.assign(
        verified_frame_count=verified_frame_counts.astype("Int64"),
        temporal_grounding_candidate=temporal_grounding_candidate.astype("boolean"),
        temporal_supervision_level=pd.Series(temporal_supervision_level,
                                            index=segment_manifest.index, dtype="string"),
        verified_temporal_boundary_available=False,
    )


Processing identity helpers are defined here, before the single calculation in Section 11.


In [13]:
def _s5_core_code_value(code):
    """Describe executable code without filenames, source lines or debug positions."""
    fields = (
        "co_argcount", "co_posonlyargcount", "co_kwonlyargcount", "co_nlocals",
        "co_stacksize", "co_flags", "co_code", "co_consts", "co_names", "co_varnames",
        "co_name", "co_qualname", "co_cellvars", "co_freevars", "co_exceptiontable",
    )
    return {field: _s5_core_value(getattr(code, field)) for field in fields if hasattr(code, field)}


def _s5_core_value(value, active=None):
    """Canonicalize scalar/container code values and defaults without changing them."""
    if isinstance(value, CodeType):
        return {"code": _s5_core_code_value(value)}
    if value is Ellipsis:
        return {"ellipsis": True}
    if value is None or isinstance(value, (str, bool, int)):
        return value
    if isinstance(value, float):
        return {"float": value.hex()} if math.isfinite(value) else {"float": repr(value)}
    if isinstance(value, complex):
        return {"complex": [_s5_core_value(value.real), _s5_core_value(value.imag)]}
    if isinstance(value, bytes):
        return {"bytes": value.hex()}
    if not isinstance(value, (dict, list, tuple, set, frozenset)):
        raise TypeError(f"Unsupported active core identity value: {type(value).__name__}")
    active = set() if active is None else active
    identity = id(value)
    if identity in active:
        raise ValueError("Cyclic containers cannot define a deterministic core identity.")
    nested = active | {identity}
    if isinstance(value, dict):
        pairs = [[_s5_core_value(key, nested), _s5_core_value(item, nested)] for key, item in value.items()]
        return {"dict": sorted(pairs, key=lambda pair: _s5_core_identity_json(pair[0]))}
    items = [_s5_core_value(item, nested) for item in value]
    if isinstance(value, (set, frozenset)):
        items = sorted(items, key=_s5_core_identity_json)
    return {type(value).__name__: items}


def _s5_core_identity_json(value):
    return json.dumps(value, sort_keys=True, ensure_ascii=False, separators=(",", ":"), allow_nan=False)


def _s5_core_referenced_names(code):
    """Include globals referenced inside comprehensions and other nested code."""
    children = (item for item in code.co_consts if isinstance(item, CodeType))
    return set(code.co_names).union(*( _s5_core_referenced_names(child) for child in children))


def _s5_core_function_value(function):
    closure = tuple(cell.cell_contents for cell in (function.__closure__ or ()))
    return {
        "code": _s5_core_code_value(function.__code__),
        "defaults": _s5_core_value(function.__defaults__),
        "kwdefaults": _s5_core_value(function.__kwdefaults__),
        "closure": _s5_core_value(closure),
    }


def sector5_runtime_core_sha256(entrypoint, namespace, dependency_versions):
    """Fingerprint active core functions, referenced S3 constants and library versions.

    Only Python functions reachable through executable global references and
    referenced ``S5_*`` constants participate. Unrelated configuration, clocks and
    notebook state are ignored. Recursive function graphs terminate through a
    visited-name set; repeated cell executions keep the same identity.
    """
    if not isinstance(entrypoint, FunctionType) or not isinstance(namespace, dict):
        raise TypeError("Provide a Python entrypoint function and its explicit namespace dictionary.")
    if not isinstance(dependency_versions, dict):
        raise TypeError("Dependency versions must be an explicit dictionary.")
    pending = {"entrypoint": entrypoint}
    functions, constants = {}, {}
    while pending:
        name = min(pending)
        function = pending.pop(name)
        if name in functions:
            continue
        functions[name] = _s5_core_function_value(function)
        for reference in sorted(_s5_core_referenced_names(function.__code__)):
            value = namespace.get(reference)
            if isinstance(value, FunctionType) and reference not in functions:
                pending[reference] = value
            elif reference.isupper() and reference in namespace and isinstance(value, (str, bool, int, float, bytes, dict, list, tuple, set, frozenset, type(None))):
                constants[reference] = _s5_core_value(value)
    payload = {
        "schema_version": 1, "functions": functions, "constants": constants,
        "dependency_versions": _s5_core_value(dependency_versions),
    }
    return hashlib.sha256(_s5_core_identity_json(payload).encode("utf-8")).hexdigest()


def _s5_core_entrypoint(labelled, config):
    """Describe the exact supplied function graph for a reproducible active-core hash."""
    grouped = add_verified_segment_numbers(labelled, config)
    grouped = add_finding_segment_ids(grouped)
    segments = build_finding_segment_manifest(grouped)
    distribution = build_split_class_distribution_report(grouped, segments)
    segments = add_temporal_supervision_metadata(segments, config)
    validate_segment_manifest_raw_references(segments, grouped)
    return {'labelled_frame_manifest_with_segments': grouped,
            'finding_segment_manifest': segments,
            'split_class_distribution_report': distribution}


def sector5_processing_signature(config, lineage):
    """Bind verified files, source table, active code and settings."""
    source_fingerprint = lineage.get("source_table_fingerprint")

    if (
        not isinstance(source_fingerprint, str)
        or re.fullmatch(r"[0-9a-f]{64}", source_fingerprint) is None
    ):
        raise RuntimeError(
            "Missing verified source-table fingerprint; "
            "rerun input loading."
        )

    versions = {
        name: importlib.metadata.version(name)
        for name in ("numpy", "pandas", "pyarrow")
    }

    core_sha = sector5_runtime_core_sha256(
        _s5_core_entrypoint, globals(), versions
    )

    payload = {
        "schema_version": 3,
        "published_source": sector5_published_source_identity(lineage),
        "published_inputs": lineage["published_inputs"],
        "raw_references": lineage["raw_references"],
        "input_sha256": lineage["input_sha256"],
        "source_table_fingerprint": source_fingerprint,
        "core_source_sha256": core_sha,
        "processing_config": {
            key: config[key]
            for key in (
                "segment_max_gap_frames",
                "minimum_verified_segment_frames",
                "storage_root",
            )
        },
    }

    signature = hashlib.sha256(
        json.dumps(
            payload,
            sort_keys=True,
            separators=(",", ":"),
            allow_nan=False,
        ).encode("utf-8")
    ).hexdigest()

    lineage["core_source_sha256"] = core_sha
    lineage["dependency_versions"] = versions

    return signature



In [14]:
def compute_sector5_result(source, config, lineage):
    """Compute only from the source verified during input loading."""
    config_snapshot = copy.deepcopy(config)
    lineage_snapshot = copy.deepcopy(lineage)
    source_snapshot = source.copy(deep=True)

    signature = sector5_processing_signature(
        config_snapshot, lineage_snapshot
    )

    source_fingerprint = sector3_table_fingerprint(source_snapshot)

    if (
        source_fingerprint
        != lineage_snapshot["source_table_fingerprint"]
    ):
        raise RuntimeError(
            "Sector 5 source table changed after verified loading. "
            "Rerun Sections 4 and 5 before computing."
        )

    # Process the exact copy whose fingerprint was verified.
    tables = _s5_core_entrypoint(source_snapshot, config_snapshot)

    if signature != sector5_processing_signature(
        config_snapshot, lineage_snapshot
    ):
        raise RuntimeError(
            "Sector 5 processing identity changed during computation; "
            "rerun Section 11."
        )

    return {
        "tables": tables,
        "processing_signature": signature,
        "source_table_fingerprint": source_fingerprint,
        "table_fingerprints": {
            name: sector3_table_fingerprint(table)
            for name, table in tables.items()
        },
    }



### 11. Compute, validate, and expose the Sector 5 results

`compute_sector5_result` invokes the existing transformation functions once, captures their processing signature, and fingerprints every output table. The following checks preserve every source annotation and frame/class anchor. Result previews follow the checks.

The segment manifest is a summary and must not replace the full annotation member table. After changing processing settings or functions, rerun this section before saving.


In [15]:
SECTOR5_RESULT = compute_sector5_result(
    source=SOURCE_LABELLED_FRAME_MANIFEST,
    config=CONFIG,
    lineage=LINEAGE,
)
SECTOR5_TABLES = SECTOR5_RESULT["tables"]
labelled_frame_manifest = SECTOR5_TABLES["labelled_frame_manifest_with_segments"]
finding_segment_manifest = SECTOR5_TABLES["finding_segment_manifest"]
split_class_distribution_report = SECTOR5_TABLES["split_class_distribution_report"]

if not finding_segment_manifest["finding_segment_id"].is_unique:
    raise RuntimeError("The segment manifest must contain one row per segment ID.")
unique_anchor_count = len(labelled_frame_manifest[
    ["video_key", "frame_number", "finding_class_normalized"]].drop_duplicates())
if int(finding_segment_manifest["verified_frame_count"].sum()) != unique_anchor_count:
    raise RuntimeError("Segment counts do not conserve the input frame/class anchors.")
if int(finding_segment_manifest["verified_annotation_count"].sum()) != len(SOURCE_LABELLED_FRAME_MANIFEST):
    raise RuntimeError("Segment counts do not conserve the input source annotations.")
source_columns = SOURCE_LABELLED_FRAME_MANIFEST.columns.tolist()
pd.testing.assert_frame_equal(
    SOURCE_LABELLED_FRAME_MANIFEST.sort_values("annotation_id").reset_index(drop=True),
    labelled_frame_manifest[source_columns].sort_values("annotation_id").reset_index(drop=True),
    check_dtype=False,
)
if labelled_frame_manifest["finding_segment_id"].nunique() != len(finding_segment_manifest):
    raise RuntimeError("Frame membership and segment manifest disagree.")
if finding_segment_manifest["verified_temporal_boundary_available"].any():
    raise RuntimeError("Observed anchors cannot be treated as verified clinical boundaries.")
display(pd.DataFrame([
    {"table": name, "rows": len(table), "columns": len(table.columns)}
    for name, table in SECTOR5_TABLES.items()
]))
print("PASS: Sector 5 transformations completed and checked.")


,table,rows,columns
0,labelled_frame_manifest_with_segments,47239,65
1,finding_segment_manifest,22220,40
2,split_class_distribution_report,42,8


PASS: Sector 5 transformations completed and checked.


In [16]:
# Result previews from the single verified calculation above.
display(labelled_frame_manifest[
    ["video_key", "finding_class_normalized", "frame_number", "opencv_frame_index",
     "segment_number", "split"]
].head(20))

print("Finding segment IDs:", labelled_frame_manifest["finding_segment_id"].nunique())

print("Verified finding segments:", f"{len(finding_segment_manifest):,}")
display(finding_segment_manifest.head())
missing_split_class_support = split_class_distribution_report.loc[
    ~split_class_distribution_report["class_present_in_split"]
]
print("Split/class combinations:", f"{len(split_class_distribution_report):,}")
print("Combinations without labelled frames:", f"{len(missing_split_class_support):,}")
display(split_class_distribution_report)
if not missing_split_class_support.empty:
    display(missing_split_class_support)

display(finding_segment_manifest[
    ["finding_segment_id", "verified_start_frame", "verified_end_frame",
     "verified_frame_count", "temporal_grounding_candidate",
     "temporal_supervision_level", "verified_temporal_boundary_available", "split"]
].head(20))
display(finding_segment_manifest["temporal_supervision_level"].value_counts(dropna=False))


,video_key,finding_class_normalized,frame_number,opencv_frame_index,segment_number,split
0,04a78ef00c5245e0,angiectasia,887,887,1,train
1,04a78ef00c5245e0,angiectasia,888,888,2,test
2,04a78ef00c5245e0,angiectasia,889,889,3,train
3,04a78ef00c5245e0,angiectasia,890,890,3,train
4,04a78ef00c5245e0,angiectasia,891,891,4,validation
5,04a78ef00c5245e0,angiectasia,892,892,5,test
6,04a78ef00c5245e0,angiectasia,893,893,6,train
7,04a78ef00c5245e0,angiectasia,894,894,6,train
8,04a78ef00c5245e0,angiectasia,895,895,6,train
9,04a78ef00c5245e0,angiectasia,896,896,6,train


Finding segment IDs: 22220
Verified finding segments: 22,220


,finding_segment_id,video_key,finding_class,finding_class_normalized,finding_class_variants_json,clinical_group,verified_start_frame,verified_end_frame,verified_frame_count,verified_annotation_count,...,target_bbox_spatially_usable,target_bbox_validation_status,target_bbox_xmin,target_bbox_ymin,target_bbox_xmax,target_bbox_ymax,split,temporal_grounding_candidate,temporal_supervision_level,verified_temporal_boundary_available
0,04a78ef00c5245e0__angiectasia__00001,04a78ef00c5245e0,Angiectasia,angiectasia,"[""Angiectasia""]",vascular_lesion,887,887,1,1,...,True,valid,245.0,2.0,279.0,22.0,train,False,single_frame_anchor_only,False
1,04a78ef00c5245e0__angiectasia__00002,04a78ef00c5245e0,Angiectasia,angiectasia,"[""Angiectasia""]",vascular_lesion,888,888,1,1,...,True,valid,233.0,1.0,272.0,23.0,test,False,single_frame_anchor_only,False
2,04a78ef00c5245e0__angiectasia__00003,04a78ef00c5245e0,Angiectasia,angiectasia,"[""Angiectasia""]",vascular_lesion,889,890,2,2,...,True,valid,255.0,12.0,290.0,42.0,train,True,multi_frame_weak_supervision,False
3,04a78ef00c5245e0__angiectasia__00004,04a78ef00c5245e0,Angiectasia,angiectasia,"[""Angiectasia""]",vascular_lesion,891,891,1,1,...,True,valid,266.0,22.0,295.0,47.0,validation,False,single_frame_anchor_only,False
4,04a78ef00c5245e0__angiectasia__00005,04a78ef00c5245e0,Angiectasia,angiectasia,"[""Angiectasia""]",vascular_lesion,892,892,1,1,...,True,valid,244.0,20.0,289.0,56.0,test,False,single_frame_anchor_only,False


Split/class combinations: 42
Combinations without labelled frames: 0


,split,finding_class_normalized,clinical_group,labelled_frame_count,video_count,segment_count,frame_fraction_within_split,class_present_in_split
0,test,normal clean mucosa,normal_mucosa,5150,37,4324,0.726683,True
1,test,ileocecal valve,anatomical_landmark,629,31,528,0.088754,True
2,test,reduced mucosal view,visibility_limitation,436,7,365,0.061521,True
3,test,pylorus,anatomical_landmark,230,30,193,0.032454,True
4,test,angiectasia,vascular_lesion,130,5,113,0.018343,True
5,test,ulcer,mucosal_lesion,128,3,116,0.018061,True
6,test,foreign body,foreign_body,116,3,98,0.016368,True
7,test,lymphangiectasia,lymphatic_lesion,89,3,75,0.012558,True
8,test,erosion,mucosal_lesion,76,9,64,0.010724,True
9,test,blood fresh,bleeding,67,2,57,0.009454,True


,finding_segment_id,verified_start_frame,verified_end_frame,verified_frame_count,temporal_grounding_candidate,temporal_supervision_level,verified_temporal_boundary_available,split
0,04a78ef00c5245e0__angiectasia__00001,887,887,1,False,single_frame_anchor_only,False,train
1,04a78ef00c5245e0__angiectasia__00002,888,888,1,False,single_frame_anchor_only,False,test
2,04a78ef00c5245e0__angiectasia__00003,889,890,2,True,multi_frame_weak_supervision,False,train
3,04a78ef00c5245e0__angiectasia__00004,891,891,1,False,single_frame_anchor_only,False,validation
4,04a78ef00c5245e0__angiectasia__00005,892,892,1,False,single_frame_anchor_only,False,test
5,04a78ef00c5245e0__angiectasia__00006,893,896,4,True,multi_frame_weak_supervision,False,train
6,04a78ef00c5245e0__angiectasia__00007,897,897,1,False,single_frame_anchor_only,False,test
7,04a78ef00c5245e0__angiectasia__00008,898,899,2,True,multi_frame_weak_supervision,False,train
8,04a78ef00c5245e0__angiectasia__00009,900,900,1,False,single_frame_anchor_only,False,validation
9,04a78ef00c5245e0__angiectasia__00010,901,901,1,False,single_frame_anchor_only,False,train


,count
temporal_supervision_level,
single_frame_anchor_only,13378
multi_frame_weak_supervision,8842


### 12. Save and verify resulted artefacts

In [17]:
# ------------------------------------------------------------------------------
# Sector 5 persistence: current catalog committed last; unchanged bytes reused.
# ------------------------------------------------------------------------------

S5_STATE_PATH = 'configs/sector5_state.json'
S5_RECEIPT_PATH = 'configs/sector5_publication_receipt.json'
S5_RESULT_NAMES = frozenset({'labelled_frame_manifest_with_segments', 'finding_segment_manifest'})
_p5_safe_path = _p3_safe_path
_p5_file_hash = _p3_file_hash


def _p5_write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(_p3_json_text(value), encoding='utf-8')
    if _p3_json_text(json.loads(path.read_text(encoding='utf-8'))) != _p3_json_text(value):
        raise RuntimeError(f'Sector 5 JSON readback failed: {path.name}')


def _p5_assert_no_secrets(value, path='configuration'):
    """Save credential references, never actual credential material."""
    if isinstance(value, dict):
        for key, item in value.items():
            if str(key).lower() in {'token', 'access_token', 'github_token', 'password',
                                   'api_key', 'authorization', 'secret', 'private_key', 'client_secret'} and item:
                raise ValueError(f'Credential material cannot be saved at {path}.{key}; use token_secret_name.')
            _p5_assert_no_secrets(item, f'{path}.{key}')
    elif isinstance(value, (list, tuple)):
        for item in value:
            _p5_assert_no_secrets(item, path)
    elif isinstance(value, str) and (
        re.search(r'https?://[^/\s]*:[^/@\s]+@', value)
        or re.search(r'(?:gh[pousr]_[A-Za-z0-9_]{20,}|github_pat_[A-Za-z0-9_]{20,})', value)
    ):
        raise ValueError(f'Credential material cannot be saved at {path}.')


def sector5_semantic_config_fingerprint(config):
    # Sector 4's projection already excludes clock metadata recursively.
    return _s3_semantic_digest(sector4_semantic_config(config))


def _p5_root(config):
    declared = Path(config.get('sector5', {}).get('local_output_dir', 'outputs/phase2/sector5')).expanduser()
    return (declared if declared.is_absolute() else Path(config['storage_root']) / declared).resolve()


def _p5_validate_lineage(lineage):
    """Recheck every consumed upstream file immediately before publishing local results."""
    sector5_published_source_identity(lineage)
    hashes, descriptors = lineage.get('input_sha256'), lineage.get('input_descriptors')
    evidence = lineage.get("published_inputs")
    if not isinstance(evidence, dict) or set(evidence) != set(hashes or {}):
        raise ValueError("Sector 5 needs pinned GitHub evidence for every consumed input.")
    if not isinstance(hashes, dict) or not hashes or not isinstance(descriptors, dict):
        raise ValueError('Sector 5 lineage needs input_sha256 and input_descriptors from checked input loading.')
    if set(hashes) != set(descriptors):
        raise ValueError('Sector 5 input file descriptors do not match its consumed input hashes.')
    for key, digest in hashes.items():
        if not isinstance(digest, str) or re.fullmatch(r'[0-9a-f]{64}', digest) is None:
            raise ValueError(f'Invalid consumed input SHA-256 in Sector 5 lineage: {key}')
        record = evidence[key]
        _s5_git_relative(record["repository_path"])
        if (record.get("source_sha256") != digest
                or record.get("encoding") not in {"identity", "gzip"}
                or re.fullmatch(r"[0-9a-f]{64}", str(record.get("export_sha256", ""))) is None):
            raise RuntimeError(f"Consumed input differs from its published GitHub evidence: {key}")
        path = Path(descriptors[key]).expanduser()
        if not path.is_absolute() or not path.is_file() or path.is_symlink():
            raise RuntimeError(f'Consumed upstream artifact is unavailable or unsafe: {key}')
        if _p3_file_hash(path) != digest:
            raise RuntimeError(f'Consumed upstream artifact changed after loading: {key}; rerun input loading.')
    core_hash = lineage.get('core_source_sha256')
    if not isinstance(core_hash, str) or re.fullmatch(r'[0-9a-f]{64}', core_hash) is None:
        raise ValueError('Sector 5 lineage lacks the active transformation SHA-256; run the processing-signature cell.')


def _p5_read_state(root):
    """Validate the complete flat catalog, current config, schemas and report bytes."""
    root = Path(root).resolve()
    state_path = _p3_safe_path(root, S5_STATE_PATH)
    if not state_path.is_file():
        return None
    state = json.loads(state_path.read_text(encoding='utf-8'))
    if state.get('schema_version') != 2 or state.get('complete') is not True:
        raise RuntimeError('Sector 5 catalog is incomplete or has an unsupported schema.')
    if state.get('content_id') != _p3_json_hash(_p3_catalog_meaning(state)):
        raise RuntimeError('Sector 5 catalog fingerprint is invalid.')
    records = {record['relative_path']: record for record in state['artifacts']}
    if len(records) != len(state['artifacts']) or {S5_STATE_PATH, S5_RECEIPT_PATH} & set(records):
        raise RuntimeError('Sector 5 catalog contains duplicate or operational artifact entries.')
    for relative, record in records.items():
        path = _p3_safe_path(root, relative)
        if not path.is_file() or path.stat().st_size != record['bytes'] or _p3_file_hash(path) != record['sha256']:
            raise RuntimeError(f'Sector 5 saved artifact checksum verification failed: {relative}')
    if state['config_snapshot'] not in records:
        raise RuntimeError('Sector 5 current configuration snapshot is absent from its catalog.')
    envelope = json.loads(_p3_safe_path(root, state['config_snapshot']).read_text(encoding='utf-8'))
    if envelope.get('semantic_config_fingerprint') != state['semantic_config_fingerprint'] or (
        sector5_semantic_config_fingerprint(envelope['effective_config']) != state['semantic_config_fingerprint']
    ):
        raise RuntimeError('Sector 5 saved effective configuration disagrees with its catalog.')

    versions = state.get("dependency_versions")

    if "dependency_versions" in state and (
        not isinstance(versions, dict)
        or set(versions) != {"numpy", "pandas", "pyarrow"}
        or any(
            not isinstance(value, str) or not value.strip()
            for value in versions.values()
        )
        or versions
        != envelope.get("lineage", {}).get("dependency_versions")
    ):
        raise RuntimeError(
            "Sector 5 dependency versions disagree "
            "with the verified config snapshot."
        )
    for name, descriptor in state['tables'].items():
        if not {descriptor['relative_path'], descriptor['schema_path']}.issubset(records):
            raise RuntimeError(f'Sector 5 table/schema absent from catalog: {name}')
        schema = json.loads(_p3_safe_path(root, descriptor['schema_path']).read_text(encoding='utf-8'))
        if schema.get('artifact') != descriptor['relative_path'] or schema.get('fingerprint') != descriptor['fingerprint']:
            raise RuntimeError(f'Sector 5 saved table schema disagrees with catalog: {name}')
    summary = state['summary']
    if not {summary['summary_relative_path'], summary['report_relative_path']}.issubset(records):
        raise RuntimeError('Sector 5 summary/report is absent from its catalog.')
    saved_summary = json.loads(_p3_safe_path(root, summary['summary_relative_path']).read_text(encoding='utf-8'))
    if _p3_json_hash(saved_summary) != summary['fingerprint'] or (
        records[summary['report_relative_path']]['sha256'] != summary['report_fingerprint']
    ):
        raise RuntimeError('Sector 5 saved summary/report disagrees with its catalog.')
    for relative in state.get('obsolete_paths', []):
        if not _p3_relative(relative).startswith('results/') or relative in records:
            raise RuntimeError('Invalid Sector 5 obsolete-result deletion authorization.')
    return state


def io_load_cached_sector5(config, processing_signature):
    """Optional checked cache; no I/O writes or Git operations."""
    root = _p5_root(config)
    state = _p5_read_state(root)
    if state is None or state['processing_signature'] != processing_signature:
        return None
    tables = {name: _p3_read_table(root, descriptor) for name, descriptor in state['tables'].items()}
    validate_segment_manifest_raw_references(
        tables['finding_segment_manifest'], tables['labelled_frame_manifest_with_segments'])
    return {'tables': tables, 'state': state}


def _p5_summary(tables):
    segments = tables.get('finding_segment_manifest')
    distribution = tables.get('split_class_distribution_report')
    summary = {'phase': 2, 'sector': 5,
               'table_row_counts': {name: len(table) for name, table in sorted(tables.items())},
               'units': {'labelled_frame_manifest_with_segments': 'accepted source annotation',
                         'finding_segment_manifest': 'finding segment',
                         'split_class_distribution_report': 'split/class combination'},
               'policy': 'Observed anchor limits are not verified clinical onset/offset. Unannotated intermediate frames remain unknown.'}
    if segments is not None:
        summary['verified_frame_class_memberships'] = int(segments['verified_frame_count'].sum())
        summary['verified_annotation_count'] = int(segments['verified_annotation_count'].sum())
        summary['target_multi_annotation_segment_count'] = int(segments['target_annotation_count'].gt(1).sum())
        summary['temporal_supervision_counts'] = {
            str(level): int(count) for level, count in segments['temporal_supervision_level'].value_counts(dropna=False).items()}
        summary['temporal_grounding_candidate_count'] = int(segments['temporal_grounding_candidate'].sum())
        summary['verified_temporal_boundary_count'] = int(segments['verified_temporal_boundary_available'].sum())
    if distribution is not None:
        summary['missing_split_class_support_count'] = int((~distribution['class_present_in_split']).sum())
        summary['split_class_distribution'] = distribution.astype(object).where(distribution.notna(), None).to_dict('records')
    return summary


def _p5_executive(summary):
    counts = summary['table_row_counts']
    return ('# Phase 2 — Sector 5: Finding Segments and Temporal Supervision\n\n'
            'This sector groups temporally adjacent verified frames with the same finding class within each video. '
            'It preserves physical-frame train, validation and test assignments, separates segments at observed split changes, creates reproducible segment identifiers, '
            'and selects a representative distinct verified frame while retaining every source annotation and bounding box.\n\n'
            f"The manifest contains **{counts.get('finding_segment_manifest', 0):,} finding segments** from "
            f"**{counts.get('labelled_frame_manifest_with_segments', 0):,} accepted source annotations**. "
            f"They represent **{summary.get('verified_frame_class_memberships', 0):,} distinct frame/class memberships**. "
            f"There are **{summary.get('temporal_grounding_candidate_count', 0):,} multi-frame temporal-grounding candidates** "
            f"and **{summary.get('missing_split_class_support_count', 0):,} split/class combinations without labelled support**.\n\n"
            'Observed start/end frames bound the available verified anchors; they are not exact clinical onset/offset. '
            'A multi-frame candidate provides weak temporal supervision only. Unannotated intermediate frames remain unknown. '
            'This sector saves manifests and reports; it does not decode or export image/video files.\n\n'
            'Manifest filenames remain stable. Configuration snapshots and report versions are written only when their '
            'semantic content changes; unchanged saved bytes are verified and reused.\n')


def _p5_write_table(stage, name, frame, fingerprint, stamp):
    is_result = name in S5_RESULT_NAMES
    base = f'results/{name}' if is_result else f'reports/{name}_{stamp}'
    relative = base + ('.parquet' if is_result else '.csv')
    path = stage / relative
    path.parent.mkdir(parents=True, exist_ok=True)
    if is_result:
        frame.to_parquet(path, index=False, compression='zstd')
        restored = pd.read_parquet(path)
    else:
        serialized = frame.astype('string')
        if serialized.eq(P3_NULL).fillna(False).any().any():
            raise ValueError(f'Reserved CSV null sentinel occurs in {name}.')
        serialized.fillna(P3_NULL).to_csv(path, index=False)
        restored = _p3_restore_csv(path, _p3_schema(frame))
    pd.testing.assert_frame_equal(frame.reset_index(drop=True), restored, check_exact=True, check_dtype=True)
    schema_relative = base + '_schema.json'
    _p5_write_json(stage / schema_relative, {**_p3_schema(frame), 'artifact': relative, 'fingerprint': fingerprint})
    return {'relative_path': relative, 'schema_path': schema_relative, 'fingerprint': fingerprint}


def _p5_install(stage, root, changed_paths, obsolete_paths):
    """Rollback on errors and replace the current state/catalog last."""
    if not root.exists():
        stage.rename(root)
        return
    backup = Path(tempfile.mkdtemp(prefix='.sector5-backup-', dir=root.parent))
    installed, moved = [], []
    sequence = [path for path in changed_paths if path != S5_STATE_PATH] + list(obsolete_paths) + [S5_STATE_PATH]
    try:
        for relative in dict.fromkeys(sequence):
            target = _p3_safe_path(root, relative)
            if target.exists():
                saved = backup / relative
                saved.parent.mkdir(parents=True, exist_ok=True)
                target.replace(saved)
                moved.append(relative)
            source = stage / relative
            if source.exists():
                target.parent.mkdir(parents=True, exist_ok=True)
                source.replace(target)
                installed.append(relative)
    except BaseException:
        for relative in reversed(installed):
            (root / relative).unlink(missing_ok=True)
        for relative in moved:
            target = root / relative
            target.parent.mkdir(parents=True, exist_ok=True)
            (backup / relative).replace(target)
        raise
    finally:
        shutil.rmtree(backup, ignore_errors=True)

def _p5_previous_for_save(root, recover):
    """Allow explicit recovery only for old Sector 5 catalog/data errors."""
    if not isinstance(recover, bool):
        raise TypeError("recover must be a boolean.")

    for name in ("configs", "results", "reports"):
        path = root / name
        if path.is_symlink() or (
            path.is_dir()
            and any(p.is_symlink() for p in path.rglob("*"))
        ):
            raise RuntimeError(
                f"Unsafe symlink in Sector 5 generated directory: {path}"
            )

    try:
        previous = _p5_read_state(root)

        if recover and previous is not None:
            for descriptor in previous["tables"].values():
                _p3_read_table(root, descriptor)

    except (
        ValueError,
        KeyError,
        TypeError,
        AttributeError,
        RuntimeError,
        FileNotFoundError,
    ) as exc:
        if not recover:
            raise
        return None, f"{type(exc).__name__}: {exc}"

    generated_paths_exist = any(
        (root / name).exists()
        for name in ("configs", "results", "reports")
    )

    if previous is None and generated_paths_exist:
        reason = (
            "Sector 5 state is missing but generated paths already exist."
        )
        if not recover:
            raise RuntimeError(
                reason + " Use recover=True to rebuild them."
            )
        return None, reason

    return previous, None


def _p5_install_rebuilt(stage, root, lineage):
    """Validate new outputs, preserve old files, and restore them on failure."""
    state = _p5_read_state(stage)
    if state is None:
        raise RuntimeError(
            "The rebuilt Sector 5 stage has no complete catalog."
        )

    for descriptor in state["tables"].values():
        _p3_read_table(stage, descriptor)

    # Install the directory containing the current catalog last.
    names = ("results", "reports", "configs")

    if any(
        not (stage / name).is_dir()
        or (stage / name).is_symlink()
        for name in names
    ):
        raise RuntimeError(
            "Invalid rebuilt Sector 5 directory layout."
        )

    if any(
        Path(path).resolve().is_relative_to(root)
        for path in lineage["input_descriptors"].values()
    ):
        raise RuntimeError(
            "Sector 5 recovery cannot contain or replace its upstream inputs."
        )

    # Recheck upstream before moving any existing output.
    _p5_validate_lineage(lineage)

    backup = Path(
        tempfile.mkdtemp(
            prefix=".sector5-recovery-",
            dir=root.parent,
        )
    )
    moved, installed = [], []

    try:
        for name in names:
            target = root / name

            if target.is_symlink():
                raise RuntimeError(
                    f"Unsafe recovery target: {target}"
                )

            if target.exists():
                target.rename(backup / name)
                moved.append(name)

        for name in names:
            (stage / name).rename(root / name)
            installed.append(name)

        _p5_read_state(root)
        for descriptor in state["tables"].values():
            _p3_read_table(root, descriptor)

    except BaseException:
        try:
            for name in reversed(installed):
                (root / name).rename(stage / name)

            for name in reversed(moved):
                (backup / name).rename(root / name)

        except BaseException as rollback_error:
            raise RuntimeError(
                "Sector 5 rollback was incomplete. "
                f"Preserve the backup at {backup}."
            ) from rollback_error

        raise

    return str(backup)

def _io_persist_sector5_verified(
    tables, config, lineage, signature, *, recover=False
):
    """Save/read back Sector 5 outputs; leave all saved files untouched when unchanged."""
    if not isinstance(signature, str) or re.fullmatch(r'[0-9a-f]{64}', signature) is None:
        raise ValueError('Pass the explicit Sector 5 SHA-256 processing signature.')
    if not isinstance(tables, dict) or not tables or any(not isinstance(frame, pd.DataFrame) for frame in tables.values()):
        raise TypeError('Sector 5 output tables must be a nonempty dictionary of DataFrames.')
    if set(tables) != {'labelled_frame_manifest_with_segments', 'finding_segment_manifest', 'split_class_distribution_report'}:
        raise ValueError('Pass all three validated Sector 5 output tables.')
    if signature != sector5_processing_signature(config, lineage):
        raise ValueError('The Sector 5 processing signature is stale; rerun its signature cell.')
    validate_segment_manifest_raw_references(
        tables['finding_segment_manifest'], tables['labelled_frame_manifest_with_segments'])
    _p5_assert_no_secrets(config)
    _p5_assert_no_secrets(lineage)
    _p5_validate_lineage(lineage)
    root = _p5_root(config)
    previous, recovery_reason = _p5_previous_for_save(root, recover)
    config_hash = sector5_semantic_config_fingerprint(config)
    provenance = {
        "input_sha256": copy.deepcopy(lineage["input_sha256"]),
        "core_source_sha256": lineage["core_source_sha256"],
        "dependency_versions": copy.deepcopy(
            lineage["dependency_versions"]
        ),
    }
    fingerprints = {name: sector3_table_fingerprint(table) for name, table in tables.items()}
    summary = _p5_summary(tables)
    summary_hash = _p3_json_hash(summary)
    report_text = _p5_executive(summary)
    report_hash = hashlib.sha256(report_text.encode('utf-8')).hexdigest()
    same_tables = previous is not None and set(fingerprints) == set(previous['tables']) and all(
        previous['tables'][name]['fingerprint'] == value for name, value in fingerprints.items())
    if previous and same_tables and all(
    previous.get(key) == value
    for key, value in provenance.items()
      ) and (
        previous['semantic_config_fingerprint'] == config_hash and previous['summary']['fingerprint'] == summary_hash
        and previous['processing_signature'] == signature and previous['summary']['report_fingerprint'] == report_hash
    ):
        for descriptor in previous['tables'].values():
            _p3_read_table(root, descriptor)
        return {'status': 'unchanged', 'root': str(root), 'local_root': str(root),
                'state': previous, 'artifacts': previous['artifacts'], 'changed_paths': []}
    root.parent.mkdir(parents=True, exist_ok=True)
    stage = Path(tempfile.mkdtemp(prefix='.sector5-stage-', dir=root.parent))
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    try:
        catalog = {record['relative_path']: record for record in previous['artifacts']} if previous else {}
        descriptors, obsolete = {}, set(previous.get('obsolete_paths', [])) if previous else set()
        config_snapshot = previous['config_snapshot'] if previous else None
        if (
            not previous
            or previous["semantic_config_fingerprint"] != config_hash
            or previous["processing_signature"] != signature
            or previous.get("dependency_versions")
            != provenance["dependency_versions"]
        ):
            config_snapshot = f'configs/sector5_config_{stamp}.json'
            _p5_write_json(stage / config_snapshot, {'schema_version': 2, 'changed_at_utc': stamp,
                'semantic_config_fingerprint': config_hash, 'effective_config': config, 'lineage': lineage})
        for name, frame in sorted(tables.items()):
            old = previous['tables'].get(name) if previous else None
            if old and old['fingerprint'] == fingerprints[name]:
                _p3_read_table(root, old)
                descriptors[name] = old
            else:
                descriptors[name] = _p5_write_table(stage, name, frame, fingerprints[name], stamp)
                if old and old['relative_path'].startswith('results/') and old['relative_path'] != descriptors[name]['relative_path']:
                    for relative in (old['relative_path'], old['schema_path']):
                        obsolete.add(relative)
                        catalog.pop(relative, None)
        previous_summary = previous['summary'] if previous else {}
        summary_path = previous_summary.get('summary_relative_path')
        if previous_summary.get('fingerprint') != summary_hash:
            summary_path = f'reports/summary_{stamp}.json'
            _p5_write_json(stage / summary_path, summary)
        report_path = previous_summary.get('report_relative_path')
        if catalog.get(report_path, {}).get('sha256') != report_hash:
            report_path = f'reports/executive_report_{stamp}.md'
            report_file = stage / report_path
            report_file.parent.mkdir(parents=True, exist_ok=True)
            report_file.write_text(report_text, encoding='utf-8')
        summary_descriptor = {'fingerprint': summary_hash, 'summary_relative_path': summary_path,
                              'report_fingerprint': report_hash, 'report_relative_path': report_path}
        changed = []
        for path in sorted(stage.rglob('*')):
            if path.is_file():
                relative = path.relative_to(stage).as_posix()
                record = {'relative_path': relative, 'sha256': _p3_file_hash(path), 'bytes': path.stat().st_size}
                if catalog.get(relative) == record:
                    path.unlink()
                else:
                    catalog[relative] = record
                    changed.append(relative)
        obsolete.difference_update(catalog)
        state = {'schema_version': 2, 'complete': True, 'processing_signature': signature,
                 'semantic_config_fingerprint': config_hash, 'config_snapshot': config_snapshot,
                 **provenance, 'tables': descriptors, 'summary': summary_descriptor,
                 'artifacts': [catalog[path] for path in sorted(catalog)], 'obsolete_paths': sorted(obsolete)}
        state['content_id'] = _p3_json_hash(_p3_catalog_meaning(state))
        state['changed_at_utc'] = stamp
        _p5_write_json(stage / S5_STATE_PATH, state)
        changed.append(S5_STATE_PATH)
        _p5_validate_lineage(lineage)

        recovery_backup_dir = None

        if recovery_reason is not None:
            recovery_backup_dir = _p5_install_rebuilt(
                stage, root, lineage
            )
        else:
            _p5_install(stage, root, changed, sorted(obsolete))
            _p5_read_state(root)

            for descriptor in state["tables"].values():
                _p3_read_table(root, descriptor)

        return {
            "status": (
                "recovered"
                if recovery_reason is not None
                else "updated"
            ),
            "root": str(root),
            "local_root": str(root),
            "state": state,
            "artifacts": state["artifacts"],
            "changed_paths": changed,
            "recovery_reason": recovery_reason,
            "recovery_backup_dir": recovery_backup_dir,
        }
    finally:
        shutil.rmtree(stage, ignore_errors=True)


def io_persist_sector5(result, config, lineage, *, recover=False):
    """Reject stale or modified results before invoking the existing writer."""
    required = {
    "tables",
    "processing_signature",
    "table_fingerprints",
    "source_table_fingerprint",
    }
    if not isinstance(result, dict) or not required.issubset(result):
        raise ValueError("Pass SECTOR5_RESULT returned by compute_sector5_result().")

    config_snapshot = copy.deepcopy(config)
    lineage_snapshot = copy.deepcopy(lineage)
    current_signature = sector5_processing_signature(config_snapshot, lineage_snapshot)

    if (
    result["source_table_fingerprint"]
    != lineage_snapshot["source_table_fingerprint"]
    ):
        raise RuntimeError(
            "Sector 5 results belong to a different verified source table. "
            "Rerun Section 11."
    )

    if current_signature != result["processing_signature"]:
        raise RuntimeError(
            "Sector 5 processing settings, code, dependencies or input lineage changed. "
            "Rerun Section 11 before saving."
        )

    current_fingerprints = {
        name: sector3_table_fingerprint(table)
        for name, table in result["tables"].items()
    }
    if current_fingerprints != result["table_fingerprints"]:
        raise RuntimeError("Sector 5 tables changed after computation; rerun Section 11.")

    return _io_persist_sector5_verified(
        result["tables"],
        config_snapshot,
        lineage_snapshot,
        result["processing_signature"],
        recover=recover,
    )


In [18]:
PROCESSING_SIGNATURE = SECTOR5_RESULT["processing_signature"]

SECTOR5_PERSISTED = io_persist_sector5(
    result=SECTOR5_RESULT,
    config=CONFIG,
    lineage=LINEAGE,
    recover=True,
)

print(
    "Sector 5 local persistence:",
    SECTOR5_PERSISTED["status"],
)
print(
    "Saved and verified:",
    SECTOR5_PERSISTED["local_root"],
)

if SECTOR5_PERSISTED.get("recovery_backup_dir"):
    print(
        "Recovery reason:",
        SECTOR5_PERSISTED["recovery_reason"],
    )
    print(
        "Previous files preserved at:",
        SECTOR5_PERSISTED["recovery_backup_dir"],
    )

display(pd.DataFrame(SECTOR5_PERSISTED["artifacts"]))

Sector 5 local persistence: updated
Saved and verified: /content/drive/MyDrive/MMVQA_Clinical/outputs/phase2/sector5


,bytes,relative_path,sha256
0,32464,configs/sector5_config_20261002T182310792275Z....,8a4e272056b23c17cd33474fb4d63bc69b6935017b9a17...
1,35898,configs/sector5_config_20261008T202426127870Z....,72f4e0d56f5438f1df9349c7a4edb76906ab22c4f2b7b0...
2,34192,configs/sector5_config_20261009T001220291738Z....,f31b764eaa5aa2237558f5b68180e39ed3a51466b94fad...
3,34292,configs/sector5_config_20261009T012622063468Z....,5dc9e12f7cc48649cf81082c65993b556d1b4e9fc03b08...
4,34292,configs/sector5_config_20261009T015057090102Z....,e4a925673ea59050f0ef8680c91d8a9e1756604520d41f...
5,53383,configs/sector5_config_20261016T005616792872Z....,d69fdb53402aad8f5e2bdc01f3c9dadfb090b2b960b4e2...
6,1151,reports/executive_report_20261002T182310792275...,95327bc99304928816d8d430168ce36652824912240ff2...
7,1204,reports/executive_report_20261008T202426127870...,543b7ed6a7a24215f35cd5a734489c15721806aaeb4cbe...
8,2883,reports/split_class_distribution_report_202610...,52734124f1aaee4667fcdf1ca431891201af3d108ec523...
9,955,reports/split_class_distribution_report_202610...,f801bf703bcbea23ac5ac85d3eee113a4f7fd917442650...


### 13. GitHub sync

In [19]:
def _p5_read_github_token(secret_name):
    """Read a credential by reference; never print, save, or embed its value in a URL."""
    import os
    import re
    if not isinstance(secret_name, str) or not secret_name or re.search(r"\s", secret_name):
        raise ValueError("github.token_secret_name must be a nonempty name without whitespace.")
    token = os.environ.get(secret_name)
    if isinstance(token, str) and token.strip():
        return token.strip()
    try:
        from google.colab import userdata
    except ImportError:
        raise RuntimeError(
            f"GitHub secret {secret_name!r} is unavailable. Run from the Colab UI "
            "or provide an environment variable with this exact name."
        ) from None
    try:
        token = userdata.get(secret_name)
    except Exception as error:
        kind = type(error).__name__
        if kind == "SecretNotFoundError":
            detail = "Create this exact name in Colab Secrets or correct github.token_secret_name."
        elif kind == "NotebookAccessError":
            detail = "Enable Notebook access for this secret using the key icon in Colab."
        elif kind == "TimeoutException":
            detail = "Retry from an active Colab browser session."
        else:
            detail = "Check the exact name and Notebook access in Colab Secrets."
        raise RuntimeError(
            f"Cannot read GitHub secret {secret_name!r} ({kind}). {detail} "
            "The saved Sector 5 results can be reused."
        ) from None
    if not isinstance(token, str) or not token.strip():
        raise RuntimeError(f"GitHub secret {secret_name!r} is empty or invalid.")
    return token.strip()


def _p5_git_options(config):
    """Validate inherited repository settings and confine exports to Sector 5."""
    import re
    from pathlib import Path, PurePosixPath
    from urllib.parse import unquote, urlsplit
    github, sector = config.get("github", {}), config.get("sector5", {})
    if not isinstance(github, dict) or not isinstance(sector, dict):
        raise TypeError("github and sector5 configuration must be dictionaries.")
    def flag(name, default):
        value = sector.get(name, github.get(name, default))
        if not isinstance(value, bool):
            raise TypeError(f"{name} must be a Python/JSON boolean.")
        return value
    push, dry_run = flag("push", True), flag("dry_run", False)
    if not push and not dry_run:
        # Local-only runs need no repository URL, author, branch validation or token.
        return {
            "github": github, "push": False, "dry_run": False,
            "branch": github.get("branch", "main"),
            "export_dir": "outputs/phase2/sector5",
            "export_parts": PurePosixPath("outputs/phase2/sector5"),
        }
    allow_local = github.get("allow_local_test_remote", sector.get("allow_local_test_remote", False))
    if not isinstance(allow_local, bool):
        raise TypeError("allow_local_test_remote must be a Python/JSON boolean.")
    export_dir = sector.get("export_dir", github.get("repo_output_dir", "outputs/phase2/sector5"))
    if not isinstance(export_dir, str) or export_dir != "outputs/phase2/sector5":
        raise ValueError("Sector 5 exports must use outputs/phase2/sector5.")
    export_parts = PurePosixPath(export_dir)
    repo_url = github.get("repo_url")
    if not isinstance(repo_url, str) or not repo_url or repo_url != repo_url.strip():
        raise ValueError("github.repo_url must be configured.")
    parsed = urlsplit(repo_url)
    github_remote = (
        parsed.scheme == "https" and parsed.netloc == "github.com"
        and parsed.username is None and parsed.password is None
        and not parsed.query and not parsed.fragment
        and re.fullmatch(r"/[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+(?:\.git)?", parsed.path) is not None
    )
    if not github_remote:
        if not allow_local:
            raise ValueError("Only credential-free HTTPS github.com repository URLs are allowed.")
        if parsed.scheme == "file" and not parsed.netloc and not parsed.query and not parsed.fragment:
            remote_path = Path(unquote(parsed.path)).resolve()
        elif not parsed.scheme and not parsed.netloc:
            remote_path = Path(repo_url).resolve()
        else:
            raise ValueError("The explicitly enabled test remote must be a local filesystem repository.")
        if not remote_path.is_dir():
            raise ValueError("The local test repository does not exist.")
        repo_url = str(remote_path)
    branch = github.get("branch", "main")
    if not isinstance(branch, str) or not branch or branch.startswith("-") or any(c in branch for c in "\r\n\x00"):
        raise ValueError("Invalid Git branch name.")
    author_name = next((github[key] for key in (
        "git_username", "git_user_name", "git_author_name", "author_name", "github_username", "user_name"
    ) if github.get(key)), None)
    author_email = next((github[key] for key in (
        "git_email", "git_user_email", "git_author_email", "author_email", "user_email"
    ) if github.get(key)), None)
    if push and not dry_run:
        if not isinstance(author_name, str) or not author_name.strip() or not isinstance(author_email, str) or not author_email.strip():
            raise ValueError("Configure the inherited Git author name and email before publishing.")
        if any(c in author_name + author_email for c in "\r\n\x00"):
            raise ValueError("Git author name and email must be single-line values.")
    return {
        "github": github, "repo_url": repo_url, "github_remote": github_remote,
        "branch": branch, "push": push, "dry_run": dry_run,
        "export_dir": export_dir, "export_parts": export_parts,
        "author_name": author_name, "author_email": author_email,
    }


def _p5_git_session(options):
    """Return a private, temporary Git session; leave existing checkouts untouched."""
    from contextlib import contextmanager
    import os
    import subprocess
    import tempfile
    from pathlib import Path
    @contextmanager
    def session():
        secret_name = options["github"].get("token_secret_name")
        token = _p5_read_github_token(secret_name) if options["github_remote"] and secret_name else None
        with tempfile.TemporaryDirectory(prefix="sector5-git-") as temporary:
            temporary_root = Path(temporary)
            checkout = temporary_root / "repository"
            checkout.mkdir()
            askpass = temporary_root / "askpass.py"
            askpass.write_text(
                "#!/usr/bin/env python3\nimport os, sys\n"
                "prompt = sys.argv[1].lower() if len(sys.argv) > 1 else ''\n"
                "sys.stdout.write('x-access-token' if 'username' in prompt else os.environ.get('_P5_GITHUB_TOKEN', ''))\n",
                encoding="utf-8",
            )
            askpass.chmod(0o700)
            hooks = temporary_root / "hooks"
            hooks.mkdir()
            child_env = {
                key: value for key, value in os.environ.items()
                if not key.startswith(("GIT_TRACE", "GIT_CONFIG_KEY_", "GIT_CONFIG_VALUE_"))
                and key not in {"GIT_CURL_VERBOSE", "SSH_ASKPASS"}
            }
            for variable in (
                "GIT_DIR", "GIT_WORK_TREE", "GIT_INDEX_FILE", "GIT_CONFIG", "GIT_CONFIG_COUNT", "GIT_COMMON_DIR",
                "GIT_OBJECT_DIRECTORY", "GIT_ALTERNATE_OBJECT_DIRECTORIES", "GIT_NAMESPACE", "_P5_GITHUB_TOKEN",
                "GIT_AUTHOR_NAME", "GIT_AUTHOR_EMAIL", "GIT_COMMITTER_NAME", "GIT_COMMITTER_EMAIL",
                "GIT_AUTHOR_DATE", "GIT_COMMITTER_DATE",
            ):
                child_env.pop(variable, None)
            child_env.update({
                "GIT_TERMINAL_PROMPT": "0", "GIT_ASKPASS": str(askpass),
                "GIT_CONFIG_NOSYSTEM": "1", "GIT_CONFIG_GLOBAL": os.devnull,
                "GIT_LITERAL_PATHSPECS": "1",
            })
            if token:
                child_env["_P5_GITHUB_TOKEN"] = token
            def git(*args, allowed=(0,)):
                result = subprocess.run(
                    ["git", "-c", "credential.helper=", "-c", f"core.askPass={askpass}",
                     "-c", f"core.hooksPath={hooks}", *args],
                    cwd=checkout, env=child_env, text=True, stdout=subprocess.PIPE,
                    stderr=subprocess.PIPE, check=False,
                )
                if result.returncode not in allowed:
                    # Do not expose remote-supplied output, which may contain credential material.
                    raise RuntimeError(f"Sector 5 Git {args[0]} failed (exit {result.returncode}).")
                return result
            git("check-ref-format", "--branch", options["branch"])
            git("init", "--quiet")
            git("remote", "add", "origin", options["repo_url"])
            yield checkout, git
    return session()


def io_git_fetch_sector5(config):
    """Fetch and report the latest remote tip without modifying the user's checkout."""
    options = _p5_git_options(config)
    if not options["push"] and not options["dry_run"]:
        return {"status": "not_requested", "branch": options["branch"]}
    with _p5_git_session(options) as (_, git):
        git("fetch", "--quiet", "origin", f"refs/heads/{options['branch']}")
        tip = git("rev-parse", "FETCH_HEAD").stdout.strip()
        return {"status": "fetched", "branch": options["branch"], "remote_tip": tip}


def io_publish_sector5(persisted, config):
    """Publish a catalog-verified Sector 5 snapshot with an ordinary non-force push.

    Only catalog artifacts, the Sector 5 state, and recorded obsolete result files
    are staged. The commit starts at the latest remote branch; races are retried.
    Existing checkouts, local edits and other pipeline sectors remain untouched.
    """
    import json
    import shutil
    from pathlib import Path, PurePosixPath
    options = _p5_git_options(config)
    branch, export_parts = options["branch"], options["export_parts"]
    outcome = {
        "status": "prepared", "branch": branch, "export_dir": options["export_dir"],
        "pushed": False, "dry_run": options["dry_run"],
    }
    if not options["push"] and not options["dry_run"]:
        return {**outcome, "status": "not_requested"}
    root_value = persisted.get("local_root", persisted.get("root"))
    if root_value is None:
        raise ValueError("The persisted Sector 5 root is missing.")
    root = Path(root_value).resolve()
    state = _p5_read_state(root)
    if state is None:
        raise RuntimeError("Sector 5 has no valid persisted snapshot to publish.")
    if persisted.get("state") != state:
        raise RuntimeError("Sector 5 changed after persistence; save it again before publishing.")
    artifacts = state.get("artifacts")
    if not isinstance(artifacts, list) or not artifacts:
        raise ValueError("Sector 5 state must contain a nonempty artifact catalog.")
    relative_paths, records = [], {}
    for artifact in artifacts:
        if not isinstance(artifact, dict):
            raise ValueError("Invalid Sector 5 artifact record.")
        relative = artifact.get("relative_path")
        source = _p5_safe_path(root, relative)
        if not source.is_file() or source.is_symlink():
            raise RuntimeError(f"Missing or unsafe Sector 5 artifact: {relative}")
        if _p5_file_hash(source) != artifact.get("sha256") or source.stat().st_size != artifact.get("bytes"):
            raise RuntimeError(f"Sector 5 artifact verification failed: {relative}")
        if relative in records:
            raise ValueError("Sector 5 artifact catalog has duplicate paths.")
        records[relative] = artifact
        relative_paths.append(relative)
    state_relative = "configs/sector5_state.json"
    state_source = _p5_safe_path(root, state_relative)
    if state_source.is_symlink() or json.loads(state_source.read_text(encoding="utf-8")) != state:
        raise RuntimeError("Sector 5 state changed during publisher verification.")
    if state_relative not in relative_paths:
        relative_paths.append(state_relative)
    obsolete_paths = state.get("obsolete_paths", [])
    if not isinstance(obsolete_paths, list) or any(not isinstance(path, str) for path in obsolete_paths):
        raise ValueError("Sector 5 obsolete_paths must be a list of recorded relative paths.")
    for relative in obsolete_paths:
        _p5_safe_path(root, relative)
        if not relative.startswith("results/") or relative in relative_paths:
            raise ValueError("Only obsolete recorded result files may be deleted from the export.")
    outcome["count"] = len(relative_paths)
    with _p5_git_session(options) as (checkout, git):
        # Each attempt starts from the newest remote branch, avoiding a hard reset/rebase.
        for attempt in range(1, 4):
            git("fetch", "--quiet", "origin", f"refs/heads/{branch}")
            remote_tip = git("rev-parse", "FETCH_HEAD").stdout.strip()
            git("checkout", "--quiet", "--detach", remote_tip)
            def export_target(relative):
                relative_path = PurePosixPath(relative)
                if relative_path.is_absolute() or any(p in {".", "..", ".git"} for p in relative_path.parts) or "\\" in relative:
                    raise ValueError("Invalid export artifact path.")
                target = checkout.joinpath(*export_parts.parts, *relative_path.parts)
                current = checkout
                for part in (*export_parts.parts, *relative_path.parts):
                    current = current / part
                    if current.is_symlink():
                        raise RuntimeError("The Git export path contains a symlink.")
                return target
            staged_paths = []
            for relative in relative_paths:
                source, target = _p5_safe_path(root, relative), export_target(relative)
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copyfile(source, target)
                if relative in records:
                    record = records[relative]
                    if _p5_file_hash(target) != record["sha256"] or target.stat().st_size != record["bytes"]:
                        raise RuntimeError("A Sector 5 artifact changed while being copied.")
                if relative == state_relative and json.loads(target.read_text(encoding="utf-8")) != state:
                    raise RuntimeError("The exported Sector 5 state changed while being copied.")
                staged_paths.append(target.relative_to(checkout).as_posix())
            for relative in obsolete_paths:
                target = export_target(relative)
                git_relative = target.relative_to(checkout).as_posix()
                tracked = bool(git("ls-files", "--", git_relative).stdout.strip())
                if target.exists():
                    if not target.is_file():
                        raise RuntimeError("An obsolete Sector 5 result path is not a file.")
                    target.unlink()
                if tracked:
                    staged_paths.append(git_relative)
            git("add", "--", *sorted(set(staged_paths)))
            changed = git("diff", "--cached", "--name-only", "-z").stdout
            changed_paths = [path for path in changed.split("\x00") if path]
            if any(not path.startswith(options["export_dir"] + "/") for path in changed_paths):
                raise RuntimeError("The publication contains a change outside Sector 5.")
            outcome.update({"remote_tip": remote_tip, "changed_paths": changed_paths, "attempt": attempt})
            if _p5_read_state(root) != state:
                raise RuntimeError("Sector 5 changed during export; save it again before publishing.")
            if not changed_paths:
                return {**outcome, "status": "unchanged"}
            if options["dry_run"]:
                return {**outcome, "status": "dry_run"}
            git("-c", f"user.name={options['author_name']}", "-c", f"user.email={options['author_email']}",
                "commit", "--quiet", "-m", "Update Phase 2 Sector 5 finding segments and temporal supervision")
            commit = git("rev-parse", "HEAD").stdout.strip()
            git("fetch", "--quiet", "origin", f"refs/heads/{branch}")
            latest_tip = git("rev-parse", "FETCH_HEAD").stdout.strip()
            if latest_tip != remote_tip:
                continue
            git("merge-base", "--is-ancestor", remote_tip, "HEAD")
            pushed = git("push", "--quiet", "origin", f"HEAD:refs/heads/{branch}", allowed=(0, 1, 128))
            if pushed.returncode == 0:
                return {**outcome, "status": "published", "pushed": True, "commit": commit}
            # Distinguish a concurrent remote update from permissions/network failures.
            git("fetch", "--quiet", "origin", f"refs/heads/{branch}")
            if git("rev-parse", "FETCH_HEAD").stdout.strip() == remote_tip:
                raise RuntimeError(
                    "Sector 5 Git push failed. Check repository access and token permissions; "
                    "the saved results remain available for retry."
                )
        raise RuntimeError("The remote branch kept changing; rerun the Sector 5 publication cell.")


In [20]:
# The saved snapshot must exist before any Git operation.
if not isinstance(SECTOR5_PERSISTED, dict) or not SECTOR5_PERSISTED.get("state"):
    raise RuntimeError("Run the Sector 5 saving cell before Git fetch.")
SECTOR5_GIT_FETCH = io_git_fetch_sector5(CONFIG)
print("Sector 5 Git fetch:", SECTOR5_GIT_FETCH)


Sector 5 Git fetch: {'status': 'fetched', 'branch': 'main', 'remote_tip': '6ae4990846104b2f7592256170f550f3b1c2b4ff'}


In [21]:
SECTOR5_PUBLICATION = io_publish_sector5(SECTOR5_PERSISTED, CONFIG)
print("Sector 5 publication:", SECTOR5_PUBLICATION["status"])
print("GitHub export:", SECTOR5_PUBLICATION["export_dir"])
print("Pushed:", SECTOR5_PUBLICATION["pushed"])
if SECTOR5_PUBLICATION.get("changed_paths"):
    display(pd.DataFrame({"changed_path": SECTOR5_PUBLICATION["changed_paths"]}))
if SECTOR5_PUBLICATION.get("commit"):
    print("Published commit:", SECTOR5_PUBLICATION["commit"])


Sector 5 publication: published
GitHub export: outputs/phase2/sector5
Pushed: True


,changed_path
0,outputs/phase2/sector5/configs/sector5_config_...
1,outputs/phase2/sector5/configs/sector5_state.json
2,outputs/phase2/sector5/results/finding_segment...
3,outputs/phase2/sector5/results/finding_segment...
4,outputs/phase2/sector5/results/labelled_frame_...
5,outputs/phase2/sector5/results/labelled_frame_...


Published commit: 0e93e5328d4d3344981468b7579add02a54dc40a
